# Phase 5 (LSI) - Model 1 (encoder-decoder, 35k BPE) + LoRMA fine-tuning + selective, relevance-based document chunking

Model 1 is an encoder-decoder LM (RoPE + GQA self-attention, RMSNorm, GELU FFN, cross-attention in the decoder) pretrained on the 35k word-level BPE. Statute identification does not need a decoder, so this notebook **fine-tunes the bidirectional encoder** (the decoder is only used once, to verify the tokenizer) with **LoRMA** (Low-Rank *Multiplicative* Adaptation), on the chunks that a **relevance selector** picks out of each long document, and puts a hierarchical document head on top.

Everything except the fine-tuning method (chunk selection, document head, loss, EMA, thresholds, resume logic, submission format) is identical to the rsLoRA version of this notebook, so the two dev tables are directly comparable.

**Kaggle setup (do this before running)**
1. *Add Data* (all found automatically under `/kaggle/input`):
   - the Phase-1 **Model-1** pretraining output (`best.pt` + `best_meta.json`; the Model-1 notebook writes no model tag, so the notebook picks the encoder-decoder `best.pt` with the lowest `val_loss`),
   - the **35k BPE** files from the BPE notebook (`vocab.json`, `merges.json`, `meta.json`).
2. *Settings*: Accelerator = GPU (T4), Internet = ON. Add a Kaggle secret named `HF_TOKEN` (a Hugging Face token with access to the gated dataset; `HF`, `HUGGINGFACE_TOKEN` also work) and tick it for this notebook. The token is read from the secret, never from the code - do not paste a token into a cell.
3. Run once with `SMOKE_TEST = True` in cell 1 (tiny subset, 1 epoch) to verify the whole pipeline and to read the real training speed, then set it to `False` and run everything.
4. Output: `/kaggle/working/p5_lsi_m1_lorma_selective_v1/submission/submission.zip` (+ `prediction.json`, `dev_probs.npy`, `test_probs.npy`, `thresholds_and_report.json`), and `selection_report.json` one level up.
5. If the session dies / the time budget ends: *Save & Commit*, attach that notebook's output as an input of the next run, run again. Training resumes automatically (adapter + head weights, merged state, optimizer, EMA, batch position, used-up training time). To train longer after a finished run, raise `TRAIN_HOURS`.

**Selective chunking (cell 6) - not random, not "first N tokens"**
1. The document is cut into **all** overlapping windows (768 tokens, stride 576) - the window length Model 1 was pretrained on.
2. Every window is **scored on CPU** (no neural model, so scoring all of them is cheap):
   `score = z( Σ evidence(token) over the window's distinct tokens ) + CITE_W · z( log(1 + #statute citations) )`.
   *Evidence(token)* = how much seeing the token raises the probability of some statute, `KL(P(l | token) ‖ P(l)) / log(1/P(l))`, learned from the training labels (so legal terms and section numbers weigh a lot, stop-words ~0). *Citations* = "section / article / act / ipc / crpc ..." followed by a number.
   Train documents are scored with statistics of the **other folds** only (a document never sees its own labels); dev and test with all-train statistics; dev labels are never used.
3. **Selected** = first window + last window (case header, operative order) + the highest-scoring windows, up to `SEL_K` (default 8), in document order. Documents with <= `SEL_K` windows use every window.
4. The document head gets each window's **original position** and its **relevance score**, so it knows where the evidence sat in the judgment.
5. An audit prints how much of the gold-statute evidence the selected windows keep, against evenly spaced / random / first+last windows with the same budget, and how much compute is saved. Set `SEL_MODE = "all"` to encode every window instead (slower, nothing dropped).
6. At prediction time `FINAL_CHUNKS = "auto"` scores dev with the selected windows, with **all** windows, and with the average of both, and uses whichever has the best cross-fitted dev macro-F1.

**LoRMA (cell 2 + 8)**
- **What it is.** LoRA *adds* a low-rank term to a frozen weight: `W = W0 + s·B·A`. LoRMA (Bihany, Patel & Modi, Findings of ACL 2025) *multiplies* the frozen weight by a low-rank-adapted matrix instead: `W = M · W0`. A purely low-rank `M` would cap the rank of `M·W0` at `r`, so the paper adds a **rank-inflation** step (the identity in LoRMA+, a permutation in LoRMA-π) and reorders the computation so the large `d_out × d_out` matrix is not needed; the update `M·W0 − W0` lives in the row space of `W0`.
- **LoRMA+ (default, `LORMA_VARIANT = "plus"`).** `M = I + s·B·A` with `B: d_out × r`, `A: r × d_out`, `s = alpha / r`. The identity is added to inflate the rank of `M`. `B = 0` at the start, so `M = I` and the encoder starts at exactly the pretrained function. It is computed as `y = W0·x + s·B(A(W0·x))`, so the `d_out × d_out` matrix `M` is never formed in the forward pass; the layer bias is kept outside the product.
- **LoRMA-π (`LORMA_VARIANT = "pi"`).** `M = P(B·A)`, where row `i` of `B·A` is cyclically shifted by `i` (a permutation instead of the identity); no `alpha`, no adapter dropout; initialised so that `M = I`. It needs a dense `d_out × d_out` multiplication in every adapted layer (for the 3072-wide `fc1` / `fc2` that is the expensive part), so expect it to be **noticeably slower** than LoRMA+. The default is LoRMA+.
- **Targets and size.** `q/k/v/o` of the self-attention and `fc1/fc2` of the feed-forward in all 6 encoder layers; the pretrained weights stay frozen. Defaults `r = 32`, `alpha = 64` (scale 2.0), `LORA_LR = 3e-4`; the log line "LoRMA+ modules: ..." prints the exact number of trainable parameters.
- **EMA and checkpoints.** The EMA is kept on the *effective* weights `M·W0` and is swapped in (adapter switched off, then restored) for evaluation; the best checkpoint stores the merged weights, so it loads without any adapter code.
- **Optional.** `RELORA = True` periodically folds `M` into `W0` and starts a fresh identity adapter (OFF by default). `FT_METHOD = "rslora"` (with `LORA_R = 64`, `LORA_ALPHA = 16`) reproduces the earlier rsLoRA run in this same notebook.
- **Not tuned.** `r`, `alpha`, `LORA_LR` and `TRAIN_HOURS` are sensible starting values, not tuned on your data. Multiplicative updates act on the whole output vector of a layer, so if the loss spikes in the first epoch, lower `LORA_LR` (e.g. `1e-4`) or `LORA_ALPHA`.

**Time**
- Training is capped by `TRAIN_HOURS = 6.5` in total over all sessions (a time-aware cosine LR schedule), with `FINAL_RESERVE_H = 1` of each session kept for dev tuning and test prediction; `EPOCHS = 5` is the upper bound.
- The speed of Model 1 on a T4 is not known in advance: run the `SMOKE_TEST` first and read the epoch ETA in the log before committing a long run.

**Things to know**
- The numbers that matter are the dev tables at the end (cross-fitted thresholds), not the training loss.
- Macro-F1 gives every statute the same weight: the loss uses a capped `pos_weight`, thresholds are tuned per statute and blended with a global one, and every document gets at least one statute.
- The tokenizer is verified by running the pretrained Model 1 on pretraining-style windows: the right BPE reproduces the pretraining `val_loss`, a wrong one gives a far larger loss and the run stops.
- The `\a-z` quirk in the BPE's word pattern (it silently drops ASCII punctuation) is reproduced on purpose - the checkpoint was pretrained on exactly that token stream.
- This is one submission from one Phase-1 model (Model 1). The task needs your best four submissions from four *different* Phase-1 models; `dev_probs.npy` / `test_probs.npy` are saved so the models can also be ensembled.

**Source:** H. Bihany, S. Patel, A. Modi, *LoRMA: Low-Rank Multiplicative Adaptation for LLMs*, Findings of ACL 2025 (arXiv 2506.07621). This notebook's implementation is written from the paper's description, not taken from the authors' code.

In [ ]:
# ============================================================
# PHASE 5 (LSI) | MODEL 1 (encoder-decoder, 35k BPE) -> its ENCODER + LoRMA fine-tuning + SELECTIVE (relevance-based) DOCUMENT CHUNKING
# CELL 1: CONFIG + HELPERS
# ============================================================
import os, re, sys, gc, json, math, time, random, shutil, zipfile, hashlib, contextlib
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

# ------------------------------------------------------------------
# SWITCH: run once with True (tiny subset, 1 epoch, ~10 min) to verify the whole pipeline end-to-end, then set False.
# ------------------------------------------------------------------
SMOKE_TEST = False

RUN_TAG = "p5_lsi_m1_lorma_selective_v1" + ("_smoke" if SMOKE_TEST else "")

IS_KAGGLE = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
WORK_DIR = ("/kaggle/working" if IS_KAGGLE else ".") + f"/{RUN_TAG}"
CKPT_DIR = f"{WORK_DIR}/checkpoints"
CACHE_DIR = f"{WORK_DIR}/cache"
OUT_DIR = f"{WORK_DIR}/submission"
for _d in (CKPT_DIR, CACHE_DIR, OUT_DIR):
    os.makedirs(_d, exist_ok=True)

# Optional manual overrides. Leave "" to auto-search /kaggle/input.
TOKENIZER_DIR = ""          # restrict the tokenizer search to this folder (the one holding vocab.json + merges.json)
PHASE1_CKPT = ""            # full path of the Model-1 Phase-1 best.pt

# ---- tokenizer: the 35k word-level BPE of the BPE notebook (vocab.json + merges.json [+ meta.json]); verified with the LM-loss check ----
TOKENIZER_CHECK = True            # run the pretrained encoder-decoder on a few documents: the right tokenizer reproduces its val_loss
TOKENIZER_STRICT = True           # raise if no candidate is close to the pretraining val_loss
LM_CHECK_DOCS = 16
LM_CHECK_MARGIN = 2.5

# ---- data ----
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE5_FILES = {
    "train": "Phase_5_LSI/Train/train.json",
    "dev":   "Phase_5_LSI/Train/dev.json",
    "test":  "Phase_5_LSI/Test/test.json",
}
SIMPLIFY_PLACEHOLDERS = True      # "<ENTITY>" -> "ENTITY" before tokenising (anonymised corpus)
MAX_DOC_TOKENS = 40_000           # pure safety cap (~70 chunks): head 70% + tail 30% are kept
KEEP_HEAD_FRAC = 0.7

# ---- overlapping chunking: the document is cut into ALL windows first, then the relevant ones are selected ----
CHUNK_LEN = 768                   # = the source window the Model-1 encoder was pretrained on
STRIDE = 576                      # 192-token overlap between consecutive chunks
OVERLAP = CHUNK_LEN - STRIDE
ENC_MB_TRAIN = 16                 # max chunks pushed through the encoder at once in training (memory guard only)
ENC_MB_EVAL = 48

# ---- SELECTIVE DOCUMENT CHUNKING --------------------------------------------------------------------------------------------
# Every chunk of every document is SCORED (cheap, CPU, no neural model); only the best SEL_K are ENCODED. Not random:
#   relevance(chunk) = z( sum over the chunk's distinct tokens of  w[token] )  +  CITE_W * z( # statute citations "section 302", "article 21" ... )
#   w[token] = how strongly the presence of the token raises the probability of some statute (information gain, each statute normalised
#              by its own entropy so rare statutes count like frequent ones); for TRAIN documents w comes from the other folds only.
#   selected = first chunk + last chunk (case header / operative order) + the highest-relevance chunks, kept in document order.
# The document head sees the ORIGINAL chunk positions and each chunk's relevance score. SEL_MODE = "all" encodes every chunk instead.
SEL_MODE = "relevance"            # "relevance" | "all"
SEL_K = 8                         # chunks encoded per document (documents with <= SEL_K chunks are used completely)
SEL_KEEP_FIRST = True
SEL_KEEP_LAST = True
SEL_FOLDS = 5                     # out-of-fold statistics for the train documents (no document scores itself with its own labels)
IG_SMOOTH = 5.0
IG_MIN_DF = 3                     # tokens seen in fewer training documents get weight 0
CITE_W = 0.5
SEL_FEAT = True                   # give the head each chunk's relevance score as an input feature
FINAL_CHUNKS = "auto"             # chunks scored for the dev/test prediction: "selected" | "all" | "both" (average of the two) | "auto" (the best of the three on dev)

# ---- fine-tuning strategy of the ENCODER of Model 1  -  THE ONE THING THAT DIFFERS from the rsLoRA notebook ----
FT_METHOD = "lorma"               # "lorma" (this notebook) | "rslora" (= the earlier Model-1 notebook: then use LORA_R = 64, LORA_ALPHA = 16) | "lora"
LORA_R = 32                       # rank of the adapters
LORA_ALPHA = 64                   # scale: lora / lorma+ = alpha / r (r = 32, alpha = 64 -> 2.0) | rslora = alpha / sqrt(r)
USE_RSLORA = (FT_METHOD == "rslora")
# LoRMA = Low-Rank Multiplicative Adaptation (Bihany, Patel & Modi, Findings of ACL 2025): the adapter MULTIPLIES the frozen weight,
#   W = M W0  with  M = I + s B A  (B: d_out x r, A: r x d_out)   instead of adding to it (LoRA: W = W0 + s B A).
LORMA_VARIANT = "plus"            # "plus" = LoRMA+ (rank inflation by adding the identity; s = alpha / r) | "pi" = LoRMA-pi (rank inflation by a cyclic row
                                  # permutation instead of the identity; no alpha, no adapter dropout)
LORA_DROPOUT = 0.05
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "fc1", "fc2")       # self-attention + feed-forward of every encoder layer
assert FT_METHOD in ("lorma", "rslora", "lora"), f"FT_METHOD must be lorma / rslora / lora, got {FT_METHOD!r}"
assert LORMA_VARIANT in ("plus", "pi"), f"LORMA_VARIANT must be plus / pi, got {LORMA_VARIANT!r}"
RELORA = False                    # optional extra, OFF: periodically merge the adapters into the weights and start new ones (works with every FT_METHOD)
RELORA_EVERY = 400                # optimizer steps between merges
RELORA_REWARM = 40                # LR of the adapters ramps up again from ~0 over this many steps after every merge (jagged cosine)
RELORA_STOP_PROG = 0.92           # no more merges in the last part of the schedule (LR is tiny there)
FT_DROPOUT = 0.1                  # dropout inside the encoder while fine-tuning (pretraining: 0.2)
GRAD_CKPT = True                  # gradient checkpointing of the encoder layers

# ---- hierarchical document head ----
HEAD_HIDDEN = 512
HEAD_DROPOUT = 0.1
DOC_LAYERS = 1
DOC_HEADS = 8
DOC_FF = 1024
MAX_POS_EMB = 128                 # learned ORIGINAL chunk-index embeddings (documents have <= ~70 chunks)
REL_BUCKETS = 8
AUX_W = 0.3                       # weight of the per-path auxiliary losses
POS_WEIGHT_CAP = 5.0              # pos_weight = clip(sqrt(neg/pos), 1, cap)

# ---- optimisation ----
EPOCHS = 10
TRAIN_TOKEN_BUDGET = 32 * CHUNK_LEN
EVAL_TOKEN_BUDGET = 96 * CHUNK_LEN
MAX_DOCS_PER_BATCH = 24
DOC_NORM = 16.0                       # loss = sum over docs / DOC_NORM  (every doc weighs the same)
LORA_LR = 3e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01                   # head weights only (adapters are re-initialised anyway)
WARMUP_STEPS = 150
MIN_LR_RATIO = 0.05
GRAD_CLIP = 1.0
USE_EMA = True
EMA_DECAY = 0.998
SAVE_EVERY = 300
LOG_EVERY = 50
SEED = 1234

# ---- time budget (free tier!) ----
TIME_BUDGET_H = 9.0                   # wall-clock limit of THIS session
FINAL_RESERVE_H = 1.0                 # kept free for dev tuning + test prediction
TRAIN_HOURS = 6.5                     # TOTAL training time over all sessions (time-aware cosine LR)

if SMOKE_TEST:
    N_SMOKE = {"train": 600, "dev": 300, "test": 200}
    EPOCHS, SAVE_EVERY, LOG_EVERY, WARMUP_STEPS = 1, 10, 5, 10
else:
    N_SMOKE = None


# ------------------------------------------------------------------
# small helpers
# ------------------------------------------------------------------
def log(*a):
    print(*a, flush=True)


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def walk_find(filename, roots):
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files:
                hits.append(d)
    return hits


HF_SECRET_NAMES = ("HF_TOKEN", "HF", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HF_API_TOKEN")      # accepted names of the secret


def get_hf_token():
    """Hugging Face token from an environment variable or from a Kaggle secret (Add-ons > Secrets) under any of HF_SECRET_NAMES"""
    for n in HF_SECRET_NAMES:
        if os.environ.get(n, "").strip():
            return os.environ[n].strip()
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
    except Exception:
        return None
    for n in HF_SECRET_NAMES:
        try:
            v = client.get_secret(n)
        except Exception:
            continue
        if v and v.strip():
            return v.strip()
    return None


T_START = time.time()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = {"enabled": False, "dtype": torch.float16}
if DEVICE.type == "cuda":
    AMP["enabled"] = True
    AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
else:
    log("WARNING: no GPU visible - this will be extremely slow")


def amp_ctx():
    """mixed precision for the encoder"""
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def no_amp():
    """full fp32 for the (small) hierarchical head: softmax / logsumexp stay exact"""
    if AMP["enabled"]:
        return torch.autocast(device_type="cuda", enabled=False)
    return contextlib.nullcontext()


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):                       # older torch
        return torch.cuda.amp.GradScaler(enabled=enabled)


# ------------------------------------------------------------------
# locate the Model-1 Phase-1 checkpoint in /kaggle/input
# ------------------------------------------------------------------
def model1_keys(path):
    """True if the checkpoint holds a Model-1 style encoder-decoder (enc_layers + dec_layers + cross attention)"""
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        keys = list(ck["model"].keys())
        ok = (any(k.endswith("enc_layers.0.attn.q_proj.weight") for k in keys)
              and any(k.endswith("dec_layers.0.cross_attn.q_proj.weight") for k in keys))
        del ck
        return ok
    except Exception:
        return False


def find_phase1_ckpt():
    """The Model-1 pretraining notebook writes best.pt / latest.pt WITHOUT a model tag (the decoder-only models of the other notebooks
    carry one). So: every best.pt / latest.pt whose meta has no tag and whose weights are an encoder-decoder; lowest val_loss wins,
    else the newest latest.pt."""
    if PHASE1_CKPT:
        return PHASE1_CKPT
    cands, seen = [], []
    for fname, mname in (("best.pt", "best_meta.json"), ("latest.pt", "latest_meta.json")):
        for d in walk_find(fname, [INPUT_ROOT]):
            meta = read_json(os.path.join(d, mname)) or {}
            path = os.path.join(d, fname)
            seen.append((path, meta.get("tag")))
            if meta.get("tag"):
                continue
            if not model1_keys(path):
                continue
            if fname == "best.pt":
                cands.append((0, float(meta.get("val_loss", 1e9)), path))
            else:
                cands.append((1, -float(meta.get("global_step", 0)), path))
    if not cands:
        raise FileNotFoundError(
            f"No Model-1 (encoder-decoder) Phase-1 checkpoint under {INPUT_ROOT}. Found: {seen or 'no best.pt/latest.pt at all'}. "
            "Attach the Model-1 pretraining output (best.pt + best_meta.json) as an input, or set PHASE1_CKPT to the full path of best.pt.")
    cands.sort()
    return cands[0][2]


assert HEAD_HIDDEN % DOC_HEADS == 0
assert SEL_MODE in ("relevance", "all") and FINAL_CHUNKS in ("selected", "all", "both", "auto")
_ft_name = {"lora": "LoRA", "rslora": "rsLoRA", "lorma": "LoRMA+" if LORMA_VARIANT == "plus" else "LoRMA-pi"}[FT_METHOD]
log("config ready | run tag:", RUN_TAG, "| smoke:", SMOKE_TEST, "| chunk selection:",
    f"{SEL_MODE} (K={SEL_K})" if SEL_MODE == "relevance" else "all chunks", "| fine-tuning:",
    f"{_ft_name} r={LORA_R}" + ("" if (FT_METHOD == "lorma" and LORMA_VARIANT == "pi") else f" alpha={LORA_ALPHA}")
    + (f" + ReLoRA merges every {RELORA_EVERY} steps" if RELORA else ""))

In [ ]:
# ============================================================
# CELL 2: MODEL  Model-1 backbone (verbatim from the Model-1 notebook) -> ENCODER only | LoRMA / rsLoRA / LoRA adapters | hierarchical document head
# Classification does not need the decoder: the (bidirectional, RoPE + GQA) encoder is fine-tuned, the decoder is dropped after the
# tokenizer check. Chunk features = [mean-pool ; max-pool] over the real tokens of a chunk (right padding is masked).
# ============================================================
@dataclass
class ModelConfig:
    vocab_size: int = 35000
    d_model: int = 768
    n_layers: int = 6
    n_heads: int = 12
    n_kv_heads: int = 4
    ffn_dim: int = 3072
    dropout: float = 0.2
    max_len: int = 1024
    pad_id: int = 0
    bos_id: int = 2
    eos_id: int = 2
    unk_id: int = 1


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd), cos/sin: (T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


def repeat_kv(x, groups):
    # (B, KV, T, hd) -> (B, KV*groups, T, hd); query head h reads kv head h // groups
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class GQASelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin, causal, pad_mask=None):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=pad_mask,
                                             is_causal=(causal and pad_mask is None))
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

    def step(self, x, k_cache, v_cache, pos, cos, sin):
        """one decoding position with a preallocated cache of shape (B, KV, max_len, hd)"""
        B = x.size(0)
        q, k, v = self._qkv(x)                                   # T == 1
        q = apply_rope(q, cos[pos:pos + 1], sin[pos:pos + 1])
        k = apply_rope(k, cos[pos:pos + 1], sin[pos:pos + 1])
        k_cache[:, :, pos] = k[:, :, 0].to(k_cache.dtype)
        v_cache[:, :, pos] = v[:, :, 0].to(v_cache.dtype)
        keys = repeat_kv(k_cache[:, :, :pos + 1], self.groups).to(q.dtype)
        vals = repeat_kv(v_cache[:, :, :pos + 1], self.groups).to(q.dtype)
        out = F.scaled_dot_product_attention(q, keys, vals)
        return self.o_proj(out.transpose(1, 2).reshape(B, 1, -1))


class CrossAttention(nn.Module):
    """plain multi-head attention, queries from the decoder, keys/values from the encoder"""

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

    def project_kv(self, mem):
        B, S, _ = mem.shape
        k = self.k_proj(mem).view(B, S, self.h, self.hd).transpose(1, 2)
        v = self.v_proj(mem).view(B, S, self.h, self.hd).transpose(1, 2)
        return k, v

    def attend(self, x, k, v, mem_mask):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k.to(q.dtype), v.to(q.dtype), attn_mask=mem_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

    def forward(self, x, mem, mem_mask):
        k, v = self.project_kv(mem)
        return self.attend(x, k, v, mem_mask)


class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.ffn_dim)
        self.fc2 = nn.Linear(cfg.ffn_dim, cfg.d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQASelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = FFN(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, mask):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin, False, mask))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.self_attn = GQASelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg)
        self.norm3 = RMSNorm(cfg.d_model)
        self.ffn = FFN(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, mem, cos, sin, mem_mask):
        x = x + self.drop(self.self_attn(self.norm1(x), cos, sin, True))
        x = x + self.drop(self.cross_attn(self.norm2(x), mem, mem_mask))
        x = x + self.drop(self.ffn(self.norm3(x)))
        return x

    def step(self, x, c, pos, mem_mask, cos, sin):
        x = x + self.self_attn.step(self.norm1(x), c["k"], c["v"], pos, cos, sin)
        x = x + self.cross_attn.attend(self.norm2(x), c["ck"], c["cv"], mem_mask)
        x = x + self.ffn(self.norm3(x))
        return x


class EncDecLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)     # shared by encoder, decoder and output head
        self.enc_layers = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.enc_norm = RMSNorm(cfg.d_model)
        self.dec_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # damp the residual-branch output projections (GPT-2 style)
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    @staticmethod
    def _mask4(src_mask):
        # (B, S) bool, True = real token  ->  (B, 1, 1, S), or None when nothing is padded
        if src_mask is None or bool(src_mask.all()):
            return None
        return src_mask[:, None, None, :]

    def encode(self, src, src_mask=None):
        mask = self._mask4(src_mask)
        x = self.drop(self.embed(src))
        for layer in self.enc_layers:
            x = layer(x, self.rope_cos, self.rope_sin, mask)
        return self.enc_norm(x)

    def decode(self, dec_in, mem, src_mask=None):
        mask = self._mask4(src_mask)
        x = self.drop(self.embed(dec_in))
        for layer in self.dec_layers:
            x = layer(x, mem, self.rope_cos, self.rope_sin, mask)
        return self.dec_norm(x)

    def forward(self, src, dec_in, src_mask=None):
        mem = self.encode(src, src_mask)
        h = self.decode(dec_in, mem, src_mask)
        return F.linear(h, self.embed.weight)          # weight tying: logits = h @ E^T

    # ---- KV-cached generation pieces ----
    def init_cache(self, mem, max_new):
        B = mem.size(0)
        kv, hd = self.cfg.n_kv_heads, self.cfg.d_model // self.cfg.n_heads
        cache = []
        for layer in self.dec_layers:
            ck, cv = layer.cross_attn.project_kv(mem)          # encoder side is projected only once
            cache.append({
                "k": mem.new_zeros(B, kv, max_new, hd),
                "v": mem.new_zeros(B, kv, max_new, hd),
                "ck": ck, "cv": cv,
            })
        return cache

    def decode_step(self, tok, pos, cache, src_mask=None):
        mask = self._mask4(src_mask)
        x = self.embed(tok[:, None])
        for layer, c in zip(self.dec_layers, cache):
            x = layer.step(x, c, pos, mask, self.rope_cos, self.rope_sin)
        x = self.dec_norm(x)
        return F.linear(x[:, 0], self.embed.weight)


P1 = {}                 # {"path", "ck", "cfg", "val_loss", "step"}   (filled in cell 3)
BACKBONES = {}


def get_backbone():
    """full Model-1 encoder-decoder with the Phase-1 weights on DEVICE (built once; needed for the tokenizer check)"""
    if "full" in BACKBONES:
        return BACKBONES["full"]
    ck = P1["ck"]
    assert "model" in ck, "Phase-1 weights were already consumed"
    sd = {re.sub(r"^(module\.|_orig_mod\.)+", "", k): v for k, v in ck["model"].items()}
    cfg = ModelConfig(**{k: v for k, v in P1["cfg"].items() if k in ModelConfig.__dataclass_fields__})
    lm = EncDecLM(cfg)
    lm.load_state_dict(sd, strict=True)               # fails loudly if architecture / keys differ
    assert CHUNK_LEN <= cfg.max_len, f"CHUNK_LEN {CHUNK_LEN} exceeds the RoPE table of the encoder ({cfg.max_len})"
    ck.pop("model", None)
    ck.pop("optimizer", None)
    del sd
    gc.collect()
    lm = lm.to(DEVICE)
    log(f"Model 1 loaded: {cfg.n_layers} encoder + {cfg.n_layers} decoder layers, d_model {cfg.d_model}, ffn {cfg.ffn_dim}, "
        f"{cfg.n_heads}q/{cfg.n_kv_heads}kv heads, vocab rows {cfg.vocab_size}, max_len {cfg.max_len}")
    BACKBONES["full"] = lm
    return lm


def to_encoder_only(lm):
    """drop the decoder (not used for classification); the embedding table is shared and stays"""
    lm.dec_layers = nn.ModuleList()
    lm.dec_norm = nn.Identity()
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return lm


# ---------------------------- fine-tuning adapters: LoRMA (multiplicative) | rsLoRA / LoRA (additive) ----------------------------
#   "lorma"   LoRMALinear   MULTIPLIES the frozen weight:  W = M W0,  M = I + s B A  (B: d_out x r, A: r x d_out)   (LoRMA+, s = alpha / r)
#                           or  M = P(B A)  (LoRMA-pi: row i of B A is cyclically shifted by i, no identity, no alpha)
#   "rslora"  LoRALinear    ADDS to it:  W = W0 + s B A,  s = alpha / sqrt(r)  (rank-stabilised)
#   "lora"    LoRALinear    the same with s = alpha / r
# Every adapter class offers the same interface to the rest of the notebook: eff_weight() (the weight it computes with), switch_off()
# (adapter = identity), adapter_clone() / adapter_restore(), merge_and_reinit() (optional ReLoRA restart).
def ft_label():
    return {"lora": "LoRA", "rslora": "rsLoRA", "lorma": "LoRMA+" if LORMA_VARIANT == "plus" else "LoRMA-pi"}[FT_METHOD]


def lora_scale(r, alpha):
    """factor the additive adapter output is multiplied with. Classic LoRA uses alpha / r, which shrinks the update more and more as the rank
    grows (a bigger r then barely helps). rsLoRA (rank-stabilised LoRA, Kalajdzievski 2023) uses alpha / sqrt(r), which keeps the
    update at a rank-independent size, so a higher rank can really be used."""
    return alpha / math.sqrt(r) if USE_RSLORA else alpha / r


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = lora_scale(r, alpha)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))      # B = 0  ->  starts exactly at the pretrained model
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

    @torch.no_grad()
    def eff_weight(self):
        """the weight this layer currently computes with: W + scale * B A"""
        return self.base.weight + self.scale * (self.B @ self.A)

    @torch.no_grad()
    def switch_off(self):
        """adapter := no change (B = 0): the layer then computes with its base weight alone"""
        self.B.zero_()

    @torch.no_grad()
    def adapter_clone(self):
        return (self.A.detach().clone(), self.B.detach().clone())

    @torch.no_grad()
    def adapter_restore(self, st):
        self.A.copy_(st[0])
        self.B.copy_(st[1])

    @torch.no_grad()
    def merge_and_reinit(self):
        """optional ReLoRA-style restart (RELORA=True): fold the adapter into the weight (the function does not change), then start a fresh adapter"""
        self.base.weight.add_(self.scale * (self.B @ self.A))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B.zero_()


_ROT_IDX = {}


def rot_index(d, device):
    """idx[i, k] = (k - i) mod d, so gather(M, 1, idx) cyclically shifts row i of M to the right by i (the LoRMA-pi permutation)"""
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device)
        _ROT_IDX[key] = (i[None, :] - i[:, None]) % d
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    """LoRMA: Low-Rank Multiplicative Adaptation. The update MULTIPLIES the frozen weight:  W = M W0  with  M = I + s B A  (B: d_out x r,
    A: r x d_out). Computed as  y = W0 x + s B (A (W0 x))  - the d_out x d_out matrix is never formed in the forward pass.
      variant "plus" (LoRMA+, rank inflation by adding the identity):  s = alpha / r, B = 0, A random      -> M = I at step 0
      variant "pi"   (LoRMA-pi, rank inflation by a permutation):      M = P(B A) where row i of B A is cyclically shifted by i; no alpha;
                     B[:, 0] = 1, A[0, 0] = 1, rest of A = 0 and the rest of B random -> P(B A) = I at step 0 (adapter dropout is not used)
    A bias of the base layer is kept as it is (y = M (W0 x) + b)."""

    def __init__(self, base, r, alpha, dropout, variant="plus"):
        super().__init__()
        assert variant in ("plus", "pi"), f"LORMA_VARIANT must be 'plus' or 'pi', got {variant!r}"
        self.base = base
        self.variant = variant
        d = base.out_features
        self.in_features, self.out_features = base.in_features, d
        r = min(r, d)
        self.r = r
        self.drop = nn.Dropout(dropout) if (dropout > 0 and variant == "plus") else nn.Identity()
        self.scale = alpha / r if variant == "plus" else 1.0
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        self._fresh()

    @torch.no_grad()
    def _fresh(self):
        """(re-)initialise the adapter so that it is the identity: M = I"""
        if self.variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B.zero_()
        else:
            nn.init.normal_(self.B, std=1.0 / math.sqrt(self.r))
            self.A.zero_()
            self.A[0, 0] = 1.0
            self.B[:, 0] = 1.0

    def mult_matrix(self):
        """the d_out x d_out matrix M of  W = M W0"""
        BA = self.B @ self.A
        if self.variant == "plus":
            return torch.eye(self.out_features, dtype=BA.dtype, device=BA.device) + self.scale * BA
        return torch.gather(BA, 1, rot_index(self.out_features, BA.device))

    @torch.no_grad()
    def eff_weight(self):
        """the weight this layer currently computes with: M W0. For LoRMA+ computed as W0 + s B (A W0), which is the same matrix
        without forming the d_out x d_out matrix M (about 50x fewer FLOPs for the 3072-wide feed-forward layer)"""
        W0 = self.base.weight
        if self.variant == "plus":
            return W0 + self.scale * (self.B @ (self.A @ W0))
        return self.mult_matrix() @ W0

    @torch.no_grad()
    def switch_off(self):
        """adapter := identity (plus: B = 0; pi: B[:, 0] = 1, A = e0 e0^T), so the layer computes with its base weight alone"""
        if self.variant == "plus":
            self.B.zero_()
        else:
            self.A.zero_()
            self.A[0, 0] = 1.0
            self.B[:, 0] = 1.0

    @torch.no_grad()
    def adapter_clone(self):
        return (self.A.detach().clone(), self.B.detach().clone())

    @torch.no_grad()
    def adapter_restore(self, st):
        self.A.copy_(st[0])
        self.B.copy_(st[1])

    @torch.no_grad()
    def merge_and_reinit(self):
        """optional ReLoRA-style restart: fold M into the weight (W0 := M W0; the function does not change), then start a fresh identity adapter"""
        self.base.weight.copy_(self.eff_weight())
        self._fresh()

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)                                    # W0 x
        if self.variant == "plus":
            y = y0 + F.linear(F.linear(self.drop(y0), self.A), self.B) * self.scale
        else:
            P = torch.gather(self.B @ self.A, 1, rot_index(self.out_features, y0.device))
            y = F.linear(y0, P)
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


ADAPTER_TYPES = (LoRALinear, LoRMALinear)


def inject_adapters(lm, targets, method, r, alpha, dropout, ft_dropout):
    """freeze the backbone, set its dropout to ft_dropout, wrap every nn.Linear whose attribute name is in `targets` with the adapter of
    `method` (lora | rslora | lorma)"""
    if method in ("lora", "rslora"):
        make = lambda child: LoRALinear(child, r, alpha, dropout)
    elif method == "lorma":
        make = lambda child: LoRMALinear(child, r, alpha, dropout, LORMA_VARIANT)
    else:
        raise ValueError(f"unknown fine-tuning method {method!r}")
    for m in lm.modules():
        if isinstance(m, nn.Dropout):
            m.p = ft_dropout
    for p in lm.parameters():
        p.requires_grad = False
    n = 0

    def wrap(module):
        nonlocal n
        for name, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and name in targets:
                setattr(module, name, make(child))
                n += 1
            elif not isinstance(child, ADAPTER_TYPES):
                wrap(child)

    wrap(lm)
    lm.enc_norm.weight.requires_grad = True          # tiny, helps adapt the pooled representation
    return n


class CkptLayer(nn.Module):
    """gradient-checkpointing wrapper around one encoder layer. Always present (only `enabled` changes), so the parameter names
    of the saved weights never depend on GRAD_CKPT."""

    def __init__(self, blk, enabled):
        super().__init__()
        self.blk = blk
        self.enabled = enabled

    def forward(self, x, cos, sin, mask):
        if self.enabled and self.training and torch.is_grad_enabled():
            return checkpoint(self.blk, x, cos, sin, mask, use_reentrant=False)
        return self.blk(x, cos, sin, mask)


# ---------------------------- hierarchical classifier over the SELECTED chunks ----------------------------
PATH_NAMES = ("attention", "chunk-LSE", "mean")


class LSIModel(nn.Module):
    """
    document = its SELECTED overlapping chunks (all of them when SEL_MODE = "all" or the document is short)
      chunk level   : encoder -> [mean-pool ; max-pool] -> MLP
                      -> + ORIGINAL chunk-index embedding + relative-position bucket (where in the document the chunk sits)
                      -> + relevance-score feature (how strongly the selector rated the chunk)
      document level: small transformer over the selected chunks (they exchange information, 'section-aware')
      3 aggregation paths, all of which use EVERY selected chunk:
        (a) label-wise attention over chunks      (b) chunk-level statute logits, log-mean-exp pooled      (c) mean of chunks
      a learned per-statute softmax gate fuses the three logits. Multi-label: independent sigmoid per statute.
    """

    def __init__(self, lm, n_labels, prior_logits):
        super().__init__()
        self.lm = lm
        d, H = lm.cfg.d_model, HEAD_HIDDEN
        self.chunk_proj = nn.Sequential(nn.LayerNorm(2 * d), nn.Dropout(HEAD_DROPOUT), nn.Linear(2 * d, H), nn.GELU())
        self.pos_emb = nn.Embedding(MAX_POS_EMB, H)           # original chunk index
        self.rel_emb = nn.Embedding(REL_BUCKETS + 1, H)       # relative position bucket (+1: single-chunk document)
        self.sc_proj = nn.Linear(1, H, bias=False)            # relevance score of the chunk (zero input when SEL_FEAT is off)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        nn.init.normal_(self.rel_emb.weight, std=0.02)
        nn.init.normal_(self.sc_proj.weight, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=H, nhead=DOC_HEADS, dim_feedforward=DOC_FF, dropout=HEAD_DROPOUT,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.doc_enc = nn.TransformerEncoder(layer, num_layers=DOC_LAYERS, enable_nested_tensor=False)
        self.doc_norm = nn.LayerNorm(H)
        self.Q = nn.Parameter(torch.randn(n_labels, H) * 0.1)              # (a) label queries over chunks
        self.W_att = nn.Parameter(torch.randn(n_labels, H) * 0.02)
        self.W_chunk = nn.Parameter(torch.randn(n_labels, H) * 0.02)       # (b) chunk-level statute classifier
        self.W_mean = nn.Parameter(torch.randn(n_labels, H) * 0.02)        # (c)
        self.gate = nn.Parameter(torch.zeros(n_labels, 3))                 # softmax -> equal mix at init
        self.b = nn.Parameter(torch.as_tensor(prior_logits, dtype=torch.float32))   # init = log prior odds

    # ---- encoder: one micro-batch of chunks -> (n, 2d) -------------------------------------
    def _encode_mb(self, ids, lens):
        N, T = ids.shape
        mask = torch.arange(T, device=ids.device)[None, :] < lens[:, None]          # (N, T) True = real token
        x = self.lm.encode(ids, mask).float()                                       # bidirectional, pad keys masked
        m = mask.unsqueeze(-1)
        mean = (x * m).sum(1) / lens[:, None].float()
        mx = x.masked_fill(~m, float("-inf")).max(1).values
        return torch.cat([mean, mx], dim=-1)

    def encode_chunks(self, ids, lens):
        """ALL chunks of the batch. Processed in length-sorted micro-batches purely to bound memory."""
        N = ids.size(0)
        mb = ENC_MB_TRAIN if self.training else ENC_MB_EVAL
        if N <= mb:
            return self._encode_mb(ids, lens)
        order = torch.argsort(lens, descending=True)
        outs = []
        for i in range(0, N, mb):
            sel = order[i:i + mb]
            T = int(lens[sel].max().item())
            outs.append(self._encode_mb(ids[sel, :T], lens[sel]))
        out = torch.cat(outs, 0)
        inv = torch.empty_like(order)
        inv[order] = torch.arange(N, device=order.device)
        return out[inv]

    # ---- document head ----------------------------------------------------------------------
    def forward(self, b):
        """b: dict from collate(). returns (logits (D,L), per-path logits incl. bias (D,L,3))"""
        n_docs, owner, slot, opos, ntot = b["nd"], b["owner"], b["slot"], b["opos"], b["ntot"]
        emb = self.encode_chunks(b["ids"], b["lens"])                         # (N, 2d) fp32
        with no_amp():
            u = self.chunk_proj(emb.float())                                  # (N, H)
            H = u.size(-1)
            rb = (((opos.float() + 0.5) / ntot.float()) * REL_BUCKETS).long().clamp(0, REL_BUCKETS - 1)
            rb = torch.where(ntot == 1, torch.full_like(rb, REL_BUCKETS), rb)
            u = u + self.pos_emb(opos.clamp(max=MAX_POS_EMB - 1)) + self.rel_emb(rb)
            u = u + self.sc_proj(b["sf"][:, None].float())

            counts = torch.bincount(owner, minlength=n_docs)                  # selected chunks per doc
            Cmax = int(counts.max().item())
            U = u.new_zeros(n_docs, Cmax, H).index_put((owner, slot), u)      # (D, C, H) padded, in document order
            valid = torch.zeros(n_docs, Cmax, dtype=torch.bool, device=u.device)
            valid[owner, slot] = True
            vm = valid[:, :, None]
            V = self.doc_norm(self.doc_enc(U, src_key_padding_mask=~valid))   # chunks attend to each other
            V = V.masked_fill(~vm, 0.0)                                       # padded slots can never leak (0 * NaN = NaN)

            nv = valid.sum(1).float()                                         # (D,)  real chunks per doc
            # (a) label-wise attention over chunks
            sc = torch.einsum("dch,lh->dcl", V, self.Q) / math.sqrt(H)
            alpha = torch.softmax(sc.masked_fill(~vm, float("-inf")), dim=1)  # (D, C, L)
            z_att = (torch.einsum("dcl,dch->dlh", alpha, V) * self.W_att).sum(-1)
            # (b) chunk-level statute logits, log-mean-exp over the real chunks
            s = torch.einsum("dch,lh->dcl", V, self.W_chunk).masked_fill(~vm, float("-inf"))
            z_pool = torch.logsumexp(s, dim=1) - torch.log(nv)[:, None]
            # (c) mean of chunk representations
            z_mean = ((V * vm).sum(1) / nv[:, None]) @ self.W_mean.t()

            zs = torch.stack([z_att, z_pool, z_mean], dim=-1)                 # (D, L, 3)
            g = torch.softmax(self.gate, dim=-1)                              # (L, 3)
            z = (zs * g[None]).sum(-1) + self.b
            return z, zs + self.b[None, :, None]


# ---------------------------- what fine-tuning changes, and how it is stored ----------------------------
def lora_modules(model):
    return [(n, m) for n, m in model.named_modules() if isinstance(m, ADAPTER_TYPES)]


def changing_names(model):
    """every parameter fine-tuning can change: the (merged) base weight of each adapted Linear, the adapters, the other trainables"""
    base_w = {f"{n}.base.weight" for n, _ in lora_modules(model)}
    return [n for n, p in model.named_parameters() if p.requires_grad or n in base_w]


def changing_state(model):
    """exact training state (for resuming): merged base weights + adapters + other trainables"""
    ps = dict(model.named_parameters())
    return {n: ps[n].detach().cpu().clone() for n in changing_names(model)}


@torch.no_grad()
def load_changing(model, sd):
    ps = dict(model.named_parameters())
    names = changing_names(model)
    missing = [n for n in names if n not in sd]
    assert not missing and set(sd) == set(names), f"checkpoint does not match the model: missing {missing[:4]} unexpected {sorted(set(sd) - set(names))[:4]}"
    for n in names:
        ps[n].copy_(sd[n].to(ps[n].device))


def merged_weights(model):
    """the function the model computes, independent of how it is split into base weight and adapter: W + scale*B*A (LoRA / rsLoRA) or
    M W0 (LoRMA) for every adapted Linear (stored under the base-weight name), plus the other trainables. This is what the best checkpoint stores."""
    ps = dict(model.named_parameters())
    out = {}
    for n, m in lora_modules(model):
        out[f"{n}.base.weight"] = m.eff_weight().detach().cpu().clone()
    for n, p in ps.items():
        if p.requires_grad and not (n.endswith(".A") or n.endswith(".B")):
            out[n] = p.detach().cpu().clone()
    return out


@torch.no_grad()
def load_merged(model, sd):
    """inverse of merged_weights(): base weight := merged weight, adapters switched off (identity)"""
    ps = dict(model.named_parameters())
    for n, v in sd.items():
        ps[n].copy_(v.to(ps[n].device))
    for _, m in lora_modules(model):
        m.switch_off()
    expected = {f"{n}.base.weight" for n, _ in lora_modules(model)} | \
               {n for n, p in ps.items() if p.requires_grad and not (n.endswith(".A") or n.endswith(".B"))}
    assert set(sd) == expected, f"merged checkpoint does not match the model: {sorted(expected ^ set(sd))[:6]}"


def lora_scale_of(model):
    """the scale the injected adapters really use (read from the first adapted layer)"""
    return lora_modules(model)[0][1].scale


def build_model():
    """Model-1 encoder + adapters of FT_METHOD (self-attention and feed-forward of every layer) + hierarchical head"""
    lm = to_encoder_only(get_backbone())
    n_lora = inject_adapters(lm, LORA_TARGETS, FT_METHOD, LORA_R, LORA_ALPHA, LORA_DROPOUT, FT_DROPOUT)
    lm.enc_layers = nn.ModuleList([CkptLayer(l, GRAD_CKPT) for l in lm.enc_layers])
    prior = np.clip(Y_train.mean(0), 1e-4, 1 - 1e-4)
    model = LSIModel(lm, N_LABELS, np.log(prior / (1 - prior))).to(DEVICE)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    if FT_METHOD == "lorma":
        how = (f"r={LORA_R}, alpha={LORA_ALPHA}, scale {lora_scale_of(model):.3f} = alpha/r, W = (I + s B A) W0" if LORMA_VARIANT == "plus"
               else f"r={LORA_R}, no alpha, W = P(B A) W0")
    else:
        how = f"r={LORA_R}, alpha={LORA_ALPHA}, scale {lora_scale(LORA_R, LORA_ALPHA):.3f} = alpha/{'sqrt(r)' if USE_RSLORA else 'r'}"
    log(f"{ft_label()} modules: {n_lora} ({how}) | trainable {n_tr / 1e6:.2f}M / total {n_all / 1e6:.1f}M params ({100 * n_tr / n_all:.1f}%)"
        + (f" | ReLoRA merges every {RELORA_EVERY} steps" if RELORA else ""))
    return model

In [ ]:
# ============================================================
# CELL 3: PHASE-1 CHECKPOINT (Model 1), DATA (HF), LABELS, DATA AUDIT
# ============================================================
from huggingface_hub import hf_hub_download, login


def load_phase1():
    """Model-1 Phase-1 pretraining checkpoint -> CPU. The architecture of the file is verified."""
    path = find_phase1_ckpt()
    ck = torch.load(path, map_location="cpu", weights_only=False)
    keys = [re.sub(r"^(module\.|_orig_mod\.)+", "", k) for k in ck["model"].keys()]
    if not (any(k.endswith("enc_layers.0.attn.q_proj.weight") for k in keys)
            and any(k.endswith("dec_layers.0.cross_attn.q_proj.weight") for k in keys)):
        raise ValueError(f"{path} is not a Model-1 encoder-decoder checkpoint (no enc_layers / dec_layers / cross-attention). "
                         "Set PHASE1_CKPT to the best.pt of the Model-1 pretraining run.")
    cfg = ck.get("config") or {}
    if hasattr(cfg, "__dataclass_fields__"):                       # config pickled as a dataclass instead of a dict
        cfg = asdict(cfg)
    cfg = dict(cfg)
    if not cfg:
        raise ValueError(f"{path} has no saved 'config' - cannot rebuild its architecture")
    P1.update({"path": path, "ck": ck, "cfg": cfg, "val_loss": ck.get("val_loss"), "step": ck.get("global_step")})
    log(f"Phase-1 Model 1: {path}")
    log(f"   config: d_model {cfg.get('d_model')} | layers {cfg.get('n_layers')}+{cfg.get('n_layers')} | heads {cfg.get('n_heads')}/{cfg.get('n_kv_heads')} kv "
        f"| vocab rows {cfg.get('vocab_size')} | max_len {cfg.get('max_len')} | pretraining val_loss {ck.get('val_loss')} | global_step {ck.get('global_step')}")


load_phase1()

# ---- Phase-5 data from Hugging Face ----
HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN)
    except Exception as e:
        log(f"WARNING: Hugging Face login failed ({type(e).__name__}) - the token is probably wrong or expired")
else:
    log("WARNING: no Hugging Face token found. The Phase-5 dataset is gated: add a Kaggle secret (Add-ons > Secrets) named HF_TOKEN, "
        "tick it for THIS notebook, and make sure that Hugging Face account has been granted access to the dataset.")


def load_records(split):
    try:
        path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE5_FILES[split],
                               repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        if type(e).__name__ in ("GatedRepoError", "RepositoryNotFoundError") or any(c in str(e) for c in ("401", "403")):
            raise RuntimeError(
                f"Cannot download {HF_DATASET_REPO} ({type(e).__name__}). " +
                ("No Hugging Face token was found: add a Kaggle secret named HF_TOKEN (Add-ons > Secrets), tick it for this notebook and re-run. "
                 if not HF_TOKEN else
                 "A token was found but it was rejected: check that it is valid and that this Hugging Face account has been granted access "
                 "to the gated dataset (open the dataset page while logged in and accept / request access). ") +
                "The token itself is read from the secret and never printed.") from e
        raise
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        for key in ("data", "records", "examples"):
            if isinstance(obj.get(key), list):
                obj = obj[key]
                break
    assert isinstance(obj, list), f"unexpected JSON structure in {split}"
    log(f"{split:5s}: {len(obj):,} documents")
    return obj


raw = {s: load_records(s) for s in ("train", "dev", "test")}


def parse_labels(lst):
    out = set()
    for x in lst:
        if isinstance(x, bool):
            continue
        if isinstance(x, (int, np.integer)):
            out.add(int(x))
        elif isinstance(x, str) and x.strip().isdigit():
            out.add(int(x.strip()))
    return sorted(out)          # sorted + unique  (3 train docs / 1 dev doc contain duplicate labels)


# label space = every statute id seen in train or dev (100 ids in your EDA)
LABEL_IDS = sorted({l for s in ("train", "dev") for r in raw[s] for l in parse_labels(r["labels"])})
LAB2IDX = {l: i for i, l in enumerate(LABEL_IDS)}
N_LABELS = len(LABEL_IDS)
log(f"{N_LABELS} statutes | id range {LABEL_IDS[0]}..{LABEL_IDS[-1]}")

if SMOKE_TEST:                  # tiny deterministic subsets (labels were computed on the full data above)
    rs = random.Random(SEED)
    for s in raw:
        keep = sorted(rs.sample(range(len(raw[s])), min(N_SMOKE[s], len(raw[s]))))
        raw[s] = [raw[s][i] for i in keep]
        log(f"SMOKE: {s} -> {len(raw[s])} docs")


def build_Y(recs):
    Y = np.zeros((len(recs), N_LABELS), dtype=np.uint8)
    for i, r in enumerate(recs):
        for l in parse_labels(r["labels"]):
            Y[i, LAB2IDX[l]] = 1
    return Y


Y_train, Y_dev = build_Y(raw["train"]), build_Y(raw["dev"])

# ---- label audit: class imbalance / rare statutes / co-occurrence ----
_pos = Y_train.sum(0)
_q = np.percentile(_pos, [0, 25, 50, 75, 100])
log(f"labels/doc: train {Y_train.sum(1).mean():.2f} | dev {Y_dev.sum(1).mean():.2f} | "
    f"single-label docs (train): {(Y_train.sum(1) == 1).mean():.1%}")
log(f"train positives per statute: min {_q[0]:.0f} | p25 {_q[1]:.0f} | median {_q[2]:.0f} | p75 {_q[3]:.0f} | max {_q[4]:.0f}")
log(f"rare statutes (train positives): <10: {int((_pos < 10).sum())} | <25: {int((_pos < 25).sum())} | "
    f"<50: {int((_pos < 50).sum())}  (macro-F1 weights each of them like the frequent ones)")
_o = np.argsort(-_pos)
log("most frequent:", ", ".join(f"{LABEL_IDS[i]}({int(_pos[i])})" for i in _o[:5]),
    "| rarest:", ", ".join(f"{LABEL_IDS[i]}({int(_pos[i])})" for i in _o[-5:]))
_C = Y_train.T.astype(np.int64) @ Y_train.astype(np.int64)
_iu = np.triu_indices(N_LABELS, 1)
_top = np.argsort(-_C[_iu])[:5]
log("top co-occurring statute pairs:", ", ".join(f"({LABEL_IDS[_iu[0][t]]},{LABEL_IDS[_iu[1][t]]}):{int(_C[_iu][t])}" for t in _top))
log(f"statutes with 0 dev positives: {int((Y_dev.sum(0) == 0).sum())}")

In [ ]:
# ============================================================
# CELL 4: TOKENIZER (the 35k word-level BPE of the BPE notebook, verified with the pretrained Model-1 LM) + TOKENISATION CACHE
# ============================================================
# ---- (A) the tokenizer of the Model-1 / BPE notebooks, extracted verbatim (word-level BPE, lower-cased, '</w>' markers).
#          NOTE: its WORD_PATTERN contains the character-class range `\a-z`, which silently DROPS ASCII punctuation
#          (. , ; : ( ) ' " - ...). The checkpoint was pretrained on exactly that token stream, so it is reproduced as is.
WORD_PATTERN = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+""")


class LegalBPE:
    def __init__(self, vocab_path, merges_path, meta_path=None):
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)
        with open(merges_path, "r", encoding="utf-8") as f:
            merges = json.load(f)
        self.ranks = {(a, b): i for i, (a, b) in enumerate(merges)}
        self.inv = {int(i): t for t, i in self.vocab.items()}
        self.vocab_size = max(self.inv) + 1
        assert len(self.inv) == self.vocab_size, "vocab ids are not contiguous"

        meta = {}
        if meta_path and os.path.exists(meta_path):
            with open(meta_path, "r") as f:
                meta = json.load(f)
        self.pad_id = self.vocab["<|pad|>"]
        self.unk_id = self.vocab["<|unk|>"]
        self.eos_id = self.vocab["<|endoftext|>"]
        for key, val in (("pad_id", self.pad_id), ("unk_id", self.unk_id), ("eos_id", self.eos_id)):
            if key in meta and meta[key] != val:
                raise ValueError(f"meta.json {key}={meta[key]} disagrees with vocab.json ({val})")
        if meta.get("vocab_size") not in (None, self.vocab_size):
            log(f"warning: meta.json vocab_size={meta['vocab_size']} but vocab.json has {self.vocab_size}")

        # no BOS in this vocab. Adding one would make the table 35001 rows, so the decoder
        # start symbol simply reuses <|endoftext|>. (EOS is never a training label here.)
        self.bos_id = self.eos_id
        self.special_ids = {self.pad_id, self.unk_id, self.eos_id}
        self.is_end = [self.inv[i].endswith("</w>") for i in range(self.vocab_size)]
        self.cache = {}

    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None:
            return hit
        syms = list(word) + ["</w>"]
        while len(syms) > 1:
            best, best_rank = None, None
            for i in range(len(syms) - 1):
                r = self.ranks.get((syms[i], syms[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = (syms[i], syms[i + 1]), r
            if best is None:
                break
            merged = best[0] + best[1]
            out, i = [], 0
            while i < len(syms):
                if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                    out.append(merged)
                    i += 2
                else:
                    out.append(syms[i])
                    i += 1
            syms = out
        ids = [self.vocab.get(s, self.unk_id) for s in syms]
        if len(self.cache) > 3_000_000:
            self.cache.clear()
        self.cache[word] = ids
        return ids

    def encode(self, text):
        ids = []
        for w in WORD_PATTERN.findall(text.lower()):
            ids.extend(self._word_ids(w))
        return ids

    def decode(self, ids):
        toks = [self.inv.get(int(i), "") for i in ids if int(i) not in self.special_ids]
        return "".join(toks).replace("</w>", " ").strip()


class CustomWordBPE(LegalBPE):
    kind = "custom-word-BPE"

    def __init__(self, vocab_path, merges_path, meta_path=None):
        super().__init__(vocab_path, merges_path, meta_path)
        h = hashlib.md5()
        for p in (vocab_path, merges_path):
            with open(p, "rb") as f:
                h.update(f.read())
        self.fingerprint = h.hexdigest()[:12]
        self.kind = "custom-word-BPE"
        self.where = os.path.dirname(vocab_path)

    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]


def _file_has_wordend_marker(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            head = f.read(2_000_000)
        return "</w>" in head
    except Exception:
        return False


def discover_tokenizers(max_rows):
    """every folder with vocab.json + merges.json (word-end markers '</w>') whose vocabulary fits the model's embedding rows"""
    roots = [TOKENIZER_DIR] if TOKENIZER_DIR else [INPUT_ROOT] + ([] if IS_KAGGLE else ["."])
    found, seen = [], set()
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            fs = set(files)
            real = os.path.realpath(d)
            if real in seen:
                continue
            seen.add(real)
            if "vocab.json" in fs and "merges.json" in fs and _file_has_wordend_marker(os.path.join(d, "vocab.json")):
                found.append(d)
    cands, fps = [], set()
    for d in found:
        try:
            t = CustomWordBPE(os.path.join(d, "vocab.json"), os.path.join(d, "merges.json"), os.path.join(d, "meta.json"))
        except Exception as e:
            log(f"  skip tokenizer in {d}: {type(e).__name__}: {str(e)[:120]}")
            continue
        if t.vocab_size > max_rows or t.vocab_size < 1000:
            log(f"  skip tokenizer in {d}: vocab {t.vocab_size} does not fit the model's {max_rows} embedding rows")
            continue
        if t.fingerprint in fps:                     # the same files attached twice
            continue
        fps.add(t.fingerprint)
        cands.append(t)
    return cands


# ---- the Phase-1 check: run the PRETRAINED encoder-decoder on pretraining-style windows ----
PRE_SRC_LEN, PRE_CONT_LEN = 768, 256                 # the Model-1 pretraining windows: encoder reads 768 tokens, decoder predicts the next 256
PRE_WIN = PRE_SRC_LEN + PRE_CONT_LEN


def cut_window(ids, is_end, s):
    """same rule as the pretraining notebook: the source/continuation cut is moved to the next word boundary (max 24 tokens)"""
    n = len(ids)
    split = s + PRE_SRC_LEN
    limit = split + 24
    while split < limit and split <= n and not is_end[ids[split - 1]]:
        split += 1
    start = split - PRE_SRC_LEN
    if start < 0 or start + PRE_WIN > n:
        return None
    return ids[start:start + PRE_WIN]


@torch.no_grad()
def lm_window_loss(lm, id_lists, is_end):
    """mean next-token CE of the PRETRAINED encoder-decoder on pretraining-style windows (source = encoder input, decoder predicts the
    256-token continuation). This is the quantity the pretraining val_loss measured: ~val_loss for the tokenizer the model was trained
    with, far larger for any other tokenizer. Windows start after the case header (token 1024 / 2304), where the text is ordinary prose."""
    lm.eval()
    tot, cnt = 0.0, 0
    rows = lm.cfg.vocab_size
    for ids in id_lists:
        wins = [w for w in (cut_window(ids, is_end, s) for s in (1024, 2304)) if w is not None] or \
               [w for w in (cut_window(ids, is_end, 0),) if w is not None]
        for w in wins:
            if max(w) >= rows:
                return float("inf"), 0
            t = torch.tensor(w, dtype=torch.long, device=DEVICE)
            src, cont = t[:PRE_SRC_LEN][None], t[PRE_SRC_LEN:]
            dec_in = torch.cat([torch.full((1,), lm.cfg.bos_id, dtype=torch.long, device=DEVICE), cont[:-1]])[None]
            with amp_ctx():
                logits = lm(src, dec_in)
            tot += F.cross_entropy(logits[0].float(), cont, ignore_index=lm.cfg.pad_id, reduction="sum").item()
            cnt += int((cont != lm.cfg.pad_id).sum().item())
    return (tot / cnt if cnt else float("nan")), cnt


# ---- text preprocessing (NO character truncation: the whole document is chunked) ----
_PH = re.compile(r"<([A-Z][A-Z_]*)>")
_WS = re.compile(r"\s+")


def prep_text(t):
    if not isinstance(t, str):
        t = ""
    if SIMPLIFY_PLACEHOLDERS:
        t = _PH.sub(r"\1", t)
    return _WS.sub(" ", t).strip()


def select_tokenizer():
    """The Model-1 BPE must be exactly the one the checkpoint was pretrained with. The check runs the pretrained encoder-decoder on
    documents tokenised by every candidate: the right tokenizer reproduces the pretraining val_loss, a wrong one gives a far larger loss."""
    lm = get_backbone()
    rows = lm.cfg.vocab_size
    cands = discover_tokenizers(rows)
    if not cands:
        raise FileNotFoundError(
            "No usable tokenizer under /kaggle/input: need vocab.json + merges.json (+ meta.json) of the 35k word-BPE from the BPE notebook "
            "(<= %d tokens). Attach the tokenizer output as an input, or set TOKENIZER_DIR." % rows)
    log("tokenizer candidates:", "; ".join(f"{t.kind} ({t.vocab_size} tokens) in {t.where}" for t in cands))
    if len(cands) == 1 and not TOKENIZER_CHECK:
        return cands[0]
    rs = random.Random(SEED)
    pool = [i for i, r in enumerate(raw["train"]) if isinstance(r.get("text"), str) and len(r["text"]) > 12000]
    pick = rs.sample(pool, min(LM_CHECK_DOCS, len(pool))) if pool else []
    texts = [prep_text(raw["train"][i]["text"]) for i in pick]
    if not texts:
        log("WARNING: no document long enough for the tokenizer check - using the first candidate")
        return cands[0]
    ref = float(P1["val_loss"]) if P1["val_loss"] is not None else 4.5
    scored = []
    for t in cands:
        loss, n = lm_window_loss(lm, t.encode_batch(texts), t.is_end)
        scored.append((loss, t))
        log(f"  {t.kind:16s} ({t.vocab_size} tokens, {t.fingerprint}) -> Model-1 next-token loss {loss:.3f} [pretraining val_loss {ref:.3f}] | {n:,} target tokens")
    scored.sort(key=lambda x: (not np.isfinite(x[0]), x[0]))
    best_loss, best = scored[0]
    if not np.isfinite(best_loss) or best_loss > ref + LM_CHECK_MARGIN:
        msg = (f"best tokenizer gives loss {best_loss:.2f} vs pretraining val_loss {ref:.2f}: the tokenizer files do not seem to be the ones "
               "this checkpoint was trained with (fine-tuning would start from noise). Attach the right tokenizer output, or set "
               "TOKENIZER_STRICT=False / TOKENIZER_CHECK=False to override.")
        if TOKENIZER_STRICT:
            raise RuntimeError(msg)
        log("WARNING:", msg)
    if len(scored) > 1:
        log(f"  -> using the tokenizer in {best.where}; the other candidate(s) are ignored")
    return best


tok = select_tokenizer() if TOKENIZER_CHECK else (discover_tokenizers(int(P1["cfg"]["vocab_size"])) or [None])[0]
assert tok is not None, "no tokenizer found"
assert tok.vocab_size <= 65535, "token ids must fit uint16"
PAD_ID = tok.pad_id
log(f"TOKENIZER IN USE: {tok.kind} | vocab={tok.vocab_size} pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id} | "
    f"fingerprint {tok.fingerprint} | {tok.where}")


# ---- tokenisation of the three splits (cached) ----
def cap_tokens(ids):
    """safety cap only (MAX_DOC_TOKENS ~ 52 chunks): keep head + tail of an absurdly long document"""
    if len(ids) <= MAX_DOC_TOKENS:
        return ids, False
    h = int(MAX_DOC_TOKENS * KEEP_HEAD_FRAC)
    return ids[:h] + ids[len(ids) - (MAX_DOC_TOKENS - h):], True


class TokStore:
    """all documents of one split as one flat uint16 array + offsets"""

    def __init__(self, tokens, offsets):
        self.tokens = tokens
        self.offsets = offsets
        self.lens = np.diff(offsets)

    def __len__(self):
        return len(self.lens)


def tokenize_split(name, recs):
    sig = {"v": 4, "n": len(recs), "tok": tok.kind, "tok_fp": tok.fingerprint, "vocab": tok.vocab_size,
           "simplify": SIMPLIFY_PLACEHOLDERS, "max_tokens": MAX_DOC_TOKENS, "head_frac": KEEP_HEAD_FRAC,
           "first": str(recs[0]["id"]), "last": str(recs[-1]["id"])}
    npz, sigp = f"{CACHE_DIR}/{name}_tokens.npz", f"{CACHE_DIR}/{name}_sig.json"
    if os.path.exists(npz) and read_json(sigp) == sig:
        z = np.load(npz)
        log(f"{name}: tokens loaded from cache")
        return TokStore(z["tokens"], z["offsets"])
    t0 = time.time()
    chunks, lens, n_capped = [], [], 0
    BS = 64
    for i in range(0, len(recs), BS):
        texts = [prep_text(r["text"]) for r in recs[i:i + BS]]
        for ids in tok.encode_batch(texts):
            if len(ids) == 0:
                ids = [tok.eos_id]
            ids, capped = cap_tokens(ids)
            n_capped += int(capped)
            chunks.append(np.asarray(ids, dtype=np.uint16))
            lens.append(len(ids))
        if (i // BS) % 40 == 0:
            log(f"  {name}: {i + len(texts):,}/{len(recs):,} docs | {(time.time() - t0) / 60:.1f} min")
    offsets = np.zeros(len(lens) + 1, dtype=np.int64)
    offsets[1:] = np.cumsum(lens)
    tokens = np.concatenate(chunks)
    np.savez(npz, tokens=tokens, offsets=offsets)
    atomic_json(sig, sigp)
    log(f"  {name}: {n_capped} docs hit the {MAX_DOC_TOKENS:,}-token safety cap")
    return TokStore(tokens, offsets)


stores = {s: tokenize_split(s, raw[s]) for s in ("train", "dev", "test")}
for s, st in stores.items():
    q = np.percentile(st.lens, [50, 90, 95, 99])
    log(f"{s:5s}: {len(st):,} docs | tokens/doc median {q[0]:.0f} p90 {q[1]:.0f} p95 {q[2]:.0f} p99 {q[3]:.0f} "
        f"max {st.lens.max():,} | total {st.lens.sum() / 1e6:.1f}M tokens")
_unk = sum(int((st.tokens == tok.unk_id).sum()) for st in stores.values()) / max(sum(len(st.tokens) for st in stores.values()), 1)
log(f"<unk> rate over all splits: {100 * _unk:.3f}%")

TEST_IDS = [r["id"] for r in raw["test"]]          # original ids, original order
for s in raw:                                      # raw texts are no longer needed
    for r in raw[s]:
        r.pop("text", None)
gc.collect()

In [ ]:
# ============================================================
# CELL 5: CHUNK GEOMETRY (all windows are generated; the selector in cell 6 decides which are encoded), METRICS, THRESHOLD TUNING  (numpy only)
# ============================================================

# ---- overlapping chunk geometry: ALL windows of a document ------------------------
def chunk_starts(n):
    """start offsets of every CHUNK_LEN window (stride STRIDE) needed to cover n tokens; the last window reaches the end"""
    return list(range(0, max(1, n - OVERLAP), STRIDE))


def n_chunks_of(n):
    return -(-max(1, n - OVERLAP) // STRIDE)


def encoded_tokens(n):
    """tokens actually pushed through the backbone for a doc of n tokens (overlap counted twice)"""
    k = n_chunks_of(n)
    return n if k == 1 else (k - 1) * CHUNK_LEN + (n - (k - 1) * STRIDE)


# ---- metrics ----------------------------------------------------------------------
def f1_per_label(Y, pred):
    Y, pred = Y.astype(bool), pred.astype(bool)
    tp = (pred & Y).sum(0).astype(np.float64)
    fp = (pred & ~Y).sum(0).astype(np.float64)
    fn = (~pred & Y).sum(0).astype(np.float64)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)


def all_metrics(Y, pred):
    Y, pred = Y.astype(bool), pred.astype(bool)
    tp, fp, fn = (pred & Y).sum(), (pred & ~Y).sum(), (~pred & Y).sum()
    return {"macro_f1": float(f1_per_label(Y, pred).mean()),
            "micro_f1": float(2 * tp / max(2 * tp + fp + fn, 1)),
            "subset_acc": float((pred == Y).all(1).mean()),
            "avg_pred_labels": float(pred.sum(1).mean())}


# ---- thresholds -------------------------------------------------------------------
GRID = np.round(np.arange(0.02, 0.96, 0.01), 2)


def f1_curves(P, Y):
    """(len(GRID), L) matrix: F1 of every label at every global threshold"""
    Yb = Y.astype(bool)
    out = np.zeros((len(GRID), P.shape[1]))
    for gi, t in enumerate(GRID):
        out[gi] = f1_per_label(Yb, P >= t)
    return out


def _smooth(curves, w=5):
    pad = w // 2
    c = np.pad(curves, ((pad, pad), (0, 0)), mode="edge")
    k = np.ones(w) / w
    return np.stack([np.convolve(c[:, j], k, mode="valid") for j in range(c.shape[1])], 1)


def fit_thresholds(P, Y, mode="blend", lam=0.5):
    """mode: global (one t) | perlabel (one t per label, smoothed F1 curve) | blend (lam*perlabel + (1-lam)*global)"""
    curves = f1_curves(P, Y)
    g = GRID[np.argmax(curves.mean(1))]
    t_l = GRID[np.argmax(_smooth(curves), 0)]
    if mode == "global":
        return np.full(P.shape[1], g)
    if mode == "perlabel":
        return t_l.astype(np.float64)
    return lam * t_l + (1 - lam) * g


def apply_thresholds(P, thr, force_top1=True):
    """multi-hot prediction; every document has >=1 statute in train/dev, so empty rows get their best-ratio label"""
    pred = P >= thr[None, :]
    if force_top1:
        empty = np.where(~pred.any(1))[0]
        if len(empty):
            pred[empty, np.argmax(P[empty] / thr[None, :], 1)] = True
    return pred


def crossfit_report(P, Y, modes=("global", "perlabel", "blend"), folds=2, seed=0):
    """honest dev estimate: thresholds fitted on the other half of dev, scored on this half"""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(P))
    parts = np.array_split(perm, folds)
    res = {}
    for mode in modes:
        pred = np.zeros(P.shape, dtype=bool)
        for k in range(folds):
            te = parts[k]
            tr = np.concatenate([parts[j] for j in range(folds) if j != k])
            thr = fit_thresholds(P[tr], Y[tr], mode)
            pred[te] = apply_thresholds(P[te], thr)
        res[mode] = all_metrics(Y, pred)
    return res


# ---- coverage audit: the windows of every document cover every token (selection happens AFTER this, in cell 6) ----
log("chunk audit (all windows of every document are generated and scored; the selector then picks which ones are encoded):")
for _s, _st in stores.items():
    _n = np.array([n_chunks_of(int(x)) for x in _st.lens])
    assert all((k - 1) * STRIDE + CHUNK_LEN >= x for k, x in zip(_n, _st.lens)), "chunk geometry leaves a gap"
    assert _n.max() <= MAX_POS_EMB, f"{_n.max()} chunks > MAX_POS_EMB={MAX_POS_EMB}"
    _enc = sum(encoded_tokens(int(x)) for x in _st.lens)
    log(f"  {_s:5s}: chunks/doc median {np.median(_n):.0f} p90 {np.percentile(_n, 90):.0f} p99 {np.percentile(_n, 99):.0f} "
        f"max {_n.max()} | total windows {_n.sum():,} | {_enc / 1e6:.1f}M tokens would go through the encoder per pass if ALL were encoded "
        f"({_enc / max(_st.lens.sum(), 1):.2f}x overlap factor)")

In [ ]:
# ============================================================
# CELL 6: SELECTIVE DOCUMENT CHUNKING  -  score EVERY window of every document, then keep the relevant ones   (numpy only)
# ============================================================
# Why not "encode everything": long judgments have dozens of windows and most of them (procedure, narration, boilerplate) say nothing about
# which statutes apply. The selector reads ALL windows (cheaply, on CPU) and spends the encoder only on those that carry statute evidence.
#
#   evidence weight of a token   w[t] = max over statutes l of  KL( P(l | t in doc) || P(l) ) / log(1/P(l))     in [0, 1]
#        (how much seeing the token raises the probability of some statute, relative to the most any token could say about that statute;
#         smoothed with IG_SMOOTH pseudo-documents, tokens seen in < IG_MIN_DF training documents get 0; stop-words get ~0 automatically)
#   chunk score  = z( sum of w over the chunk's DISTINCT tokens )  +  CITE_W * z( log(1 + # statute citations) )
#        citation = cue word ("section", "article", "act", "ipc", "crpc", ...) followed within 1-2 tokens by a number
#   selected     = first window + last window (case header / operative order) + the highest-scoring windows, up to SEL_K, in document order
#
# TRAIN documents are scored with statistics of the OTHER folds only (no document is scored with its own labels); dev/test documents with
# statistics of the whole training set. Dev labels are never used by the selector, so dev/test stay honest.
CUE_WORDS = ("section", "sections", "sec", "article", "articles", "act", "ipc", "crpc", "cpc", "rule", "rules", "s")
SEL_VERSION = 1
assert SEL_K >= int(SEL_KEEP_FIRST) + int(SEL_KEEP_LAST), "SEL_K must leave room for the forced first/last window"


def doc_tokens(store, d):
    return store.tokens[int(store.offsets[d]): int(store.offsets[d + 1])]


def vocab_flags():
    """per token id: is it a statute-citation cue word / a number piece (word-level BPE: '302</w>' or a piece '30' of a longer number)"""
    V = tok.vocab_size
    is_cue, is_num = np.zeros(V, dtype=bool), np.zeros(V, dtype=bool)
    for w in CUE_WORDS:
        i = tok.vocab.get(w + "</w>")
        if i is not None and i < V:
            is_cue[i] = True
    for i, t in tok.inv.items():
        s = t[:-4] if t.endswith("</w>") else t
        if i < V and s and s.isascii() and s.isdigit():
            is_num[i] = True
    return is_cue, is_num


def count_df(store, Y, fold_of, n_folds):
    """document frequencies per fold: dft[f, t] = #docs of fold f containing token t ; dfp[f, l, t] = ... that also carry statute l"""
    V, L = tok.vocab_size, Y.shape[1]
    dft = np.zeros((n_folds, V), dtype=np.int32)
    dfp = np.zeros((n_folds, L, V), dtype=np.int32)
    nf = np.zeros(n_folds, dtype=np.int64)
    npf = np.zeros((n_folds, L), dtype=np.int64)
    t0 = time.time()
    for d in range(len(store)):
        f = int(fold_of[d])
        u = np.unique(doc_tokens(store, d)).astype(np.int64)
        lab = np.flatnonzero(Y[d])
        dft[f, u] += 1
        if len(lab):
            dfp[f, lab[:, None], u[None, :]] += 1
        nf[f] += 1
        npf[f, lab] += 1
        if d % 5000 == 4999:
            log(f"  selector statistics: {d + 1:,}/{len(store):,} train docs | {time.time() - t0:.0f}s")
    return dft, dfp, nf, npf


def info_gain_weights(df_tot, df_pos, N, Npos, smooth, min_df):
    """-> (w (V,), W (L, V)):  W[l, t] = KL(P(l|t) || P(l)) / log(1/P(l)) for positive associations (else 0), w = max over statutes"""
    p = np.clip(np.asarray(Npos, dtype=np.float64) / max(int(N), 1), 1e-6, 1 - 1e-6)[:, None]       # (L, 1) prior
    n_t = np.asarray(df_tot, dtype=np.float64)[None, :]                                               # (1, V)
    q = (np.asarray(df_pos, dtype=np.float64) + smooth * p) / (n_t + smooth)                          # smoothed P(l | token present)
    q = np.clip(q, 1e-9, 1 - 1e-9)
    kl = q * np.log(q / p) + (1 - q) * np.log((1 - q) / (1 - p))
    kl = np.where(q > p, kl, 0.0)                                                                     # only evidence FOR the statute
    W = np.clip(kl / np.log(1.0 / p), 0.0, 1.0)                                                       # KL <= log(1/p): bounded, comparable across statutes
    W[:, np.asarray(df_tot) < min_df] = 0.0
    return W.max(0).astype(np.float32), W.astype(np.float32)


def chunk_features(x, w, is_cue, is_num):
    """x: token ids of one document. -> (evidence sum (k,), citation count (k,)) for ALL windows of the document"""
    n = len(x)
    starts = chunk_starts(n)
    xi = np.asarray(x, dtype=np.int64)
    cue, num = is_cue[xi], is_num[xi]
    hit = np.zeros(n, dtype=bool)                                     # hit[i]: cue word at i followed within 1-2 tokens by a number
    if n > 1:
        hit[:-1] |= cue[:-1] & num[1:]
    if n > 2:
        hit[:-2] |= cue[:-2] & num[2:]
    cs = np.concatenate([[0], np.cumsum(hit)])
    ig = np.empty(len(starts), dtype=np.float32)
    ci = np.empty(len(starts), dtype=np.float32)
    for j, s in enumerate(starts):
        e = min(n, s + CHUNK_LEN)
        ig[j] = w[np.unique(xi[s:e])].sum(dtype=np.float64)
        ci[j] = cs[e] - cs[s]
    return ig, ci


def select_chunks(scores, k, keep_first=True, keep_last=True):
    """indices (ascending = document order) of the windows to encode: forced first/last + the best-scoring others; ties -> earlier window"""
    n = len(scores)
    if n <= k:
        return np.arange(n, dtype=np.int64)
    forced = ([0] if keep_first else []) + ([n - 1] if keep_last else [])          # n > k >= 2 here, so 0 and n-1 differ
    m = k - len(forced)
    rest = np.array([i for i in range(n) if i not in forced], dtype=np.int64)
    top = rest[np.argsort(-np.asarray(scores)[rest], kind="stable")[:m]] if m > 0 else np.zeros(0, dtype=np.int64)
    return np.array(sorted(list(forced) + [int(i) for i in top]), dtype=np.int64)


def chunk_token_len(n, j):
    """real tokens in window j of an n-token document"""
    return min(CHUNK_LEN, n - j * STRIDE)


def compute_scores():
    """relevance score of EVERY window of EVERY document. -> (SCORE {split: [array per doc]}, W_all (L, V))"""
    is_cue, is_num = vocab_flags()
    F_ = SEL_FOLDS
    n_tr = len(stores["train"])
    fold_of = np.random.RandomState(SEED + 7).permutation(n_tr) % F_
    t0 = time.time()
    dft, dfp, nf, npf = count_df(stores["train"], Y_train, fold_of, F_)
    tot_t, tot_p, N, Np = dft.sum(0), dfp.sum(0), int(nf.sum()), npf.sum(0)
    w_all, W_all = info_gain_weights(tot_t, tot_p, N, Np, IG_SMOOTH, IG_MIN_DF)
    w_fold = [info_gain_weights(tot_t - dft[f], tot_p - dfp[f], N - int(nf[f]), Np - npf[f], IG_SMOOTH, IG_MIN_DF)[0] for f in range(F_)]
    del dft, dfp
    log(f"selector statistics ready ({time.time() - t0:.0f}s) | tokens with evidence weight > 0.05: {int((w_all > 0.05).sum()):,} of {tok.vocab_size:,} "
        f"| strongest: " + ", ".join(f"{tok.inv[int(i)].replace('</w>', '')}({w_all[i]:.2f})" for i in np.argsort(-w_all)[:6]))

    feats = {}
    for split in ("train", "dev", "test"):
        st = stores[split]
        ig_l, ci_l = [], []
        for d in range(len(st)):
            w = w_fold[int(fold_of[d])] if split == "train" else w_all
            ig, ci = chunk_features(doc_tokens(st, d), w, is_cue, is_num)
            ig_l.append(ig)
            ci_l.append(ci)
        feats[split] = (ig_l, ci_l)
        log(f"  scored the windows of {len(st):,} {split} docs | {time.time() - t0:.0f}s")
    a = np.concatenate(feats["train"][0]).astype(np.float64)
    c = np.log1p(np.concatenate(feats["train"][1]).astype(np.float64))
    mu_a, sd_a, mu_c, sd_c = a.mean(), a.std() + 1e-8, c.mean(), c.std() + 1e-8
    score = {s: [((ig - mu_a) / sd_a + CITE_W * (np.log1p(ci) - mu_c) / sd_c).astype(np.float32) for ig, ci in zip(*feats[s])] for s in feats}
    return score, W_all


def _flat(lst):
    off = np.zeros(len(lst) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(x) for x in lst])
    return (np.concatenate(lst) if len(lst) else np.zeros(0, dtype=np.float32)), off


def get_scores():
    """cached: the scoring needs the training statistics only, so a restarted session re-uses it"""
    sig = {"v": SEL_VERSION, "tok_fp": tok.fingerprint, "n": {s: len(stores[s]) for s in stores}, "chunk": [CHUNK_LEN, STRIDE],
           "folds": SEL_FOLDS, "smooth": IG_SMOOTH, "min_df": IG_MIN_DF, "cite_w": CITE_W, "seed": SEED, "cues": list(CUE_WORDS),
           "y": hashlib.md5(Y_train.tobytes()).hexdigest()[:12]}
    npz, sigp = f"{CACHE_DIR}/selector_scores.npz", f"{CACHE_DIR}/selector_sig.json"
    if os.path.exists(npz) and read_json(sigp) == sig:
        z = np.load(npz)
        score = {}
        for s in stores:
            flat, off = z[f"score_{s}"], z[f"off_{s}"]
            score[s] = [flat[off[i]:off[i + 1]] for i in range(len(off) - 1)]
        log("selector scores loaded from cache")
        return score, z["W_all"]
    score, W_all = compute_scores()
    arrs = {"W_all": W_all}
    for s in score:
        arrs[f"score_{s}"], arrs[f"off_{s}"] = _flat(score[s])
    np.savez(npz, **arrs)
    atomic_json(sig, sigp)
    return score, W_all


SCORE, W_ALL = get_scores()
for _s in stores:                       # one score per window
    assert all(len(SCORE[_s][d]) == n_chunks_of(int(stores[_s].lens[d])) for d in range(len(stores[_s]))), "score / window count mismatch"

SEL_REL = {s: [select_chunks(SCORE[s][d], SEL_K, SEL_KEEP_FIRST, SEL_KEEP_LAST) for d in range(len(stores[s]))] for s in stores}
SEL_ALL = {s: [np.arange(n_chunks_of(int(stores[s].lens[d])), dtype=np.int64) for d in range(len(stores[s]))] for s in stores}
SEL = SEL_REL if SEL_MODE == "relevance" else SEL_ALL          # what the model is TRAINED on
SFEAT = {s: [(np.clip(x / 4.0, -1.0, 1.0) if SEL_FEAT else np.zeros_like(x)).astype(np.float32) for x in SCORE[s]] for s in stores}


def sel_for(split, mode):
    """chunk indices per document for 'selected' (relevance) or 'all'"""
    return SEL_REL[split] if mode == "selected" else SEL_ALL[split]


# ---- audit 1: how much compute does the selection save ----
def selection_summary():
    rep = {}
    for s in stores:
        n = np.array([n_chunks_of(int(x)) for x in stores[s].lens])
        k = np.array([len(x) for x in SEL_REL[s]])
        enc_sel = sum(chunk_token_len(int(stores[s].lens[d]), int(j)) for d in range(len(stores[s])) for j in SEL_REL[s][d])
        enc_all = sum(chunk_token_len(int(stores[s].lens[d]), j) for d in range(len(stores[s])) for j in range(int(n[d])))
        rep[s] = {"docs": len(n), "windows": int(n.sum()), "selected": int(k.sum()), "docs_truncated_by_selection": float((n > SEL_K).mean()),
                  "mean_windows_per_doc": float(n.mean()), "mean_selected_per_doc": float(k.mean()),
                  "encoder_tokens_selected": int(enc_sel), "encoder_tokens_all": int(enc_all), "compute_fraction": float(enc_sel / max(enc_all, 1))}
        log(f"  {s:5s}: {int(n.sum()):,} windows -> {int(k.sum()):,} selected ({100 * k.sum() / n.sum():.0f}%) | docs that lose windows: "
            f"{100 * (n > SEL_K).mean():.0f}% | encoder tokens {enc_sel / 1e6:.1f}M of {enc_all / 1e6:.1f}M ({100 * enc_sel / max(enc_all, 1):.0f}% of the compute)")
    return rep


# ---- audit 2: does the selection keep the evidence? (dev labels are used HERE ONLY, to judge the selector, never to choose windows) ----
def evidence_audit(max_docs=1500):
    """evidence of a document = sum over its distinct tokens of  max_{gold statute} W[l, t]. Share of it that sits inside the selected windows,
    versus evenly spaced windows, random windows (first/last kept) and first+last only - all with the same number of windows."""
    st = stores["dev"]
    rs = np.random.RandomState(SEED + 11)
    cand = [d for d in range(len(st)) if n_chunks_of(int(st.lens[d])) > SEL_K and Y_dev[d].any()]
    if not cand:
        log("  evidence audit: no dev document has more than SEL_K windows - nothing is dropped")
        return None
    cand = [int(d) for d in rs.permutation(cand)[:max_docs]]
    res = {"relevance (used)": [], "evenly spaced": [], "random + first/last": [], "first + last only": []}
    for d in cand:
        x = doc_tokens(st, d).astype(np.int64)
        n = len(x)
        starts = chunk_starts(n)
        k = len(starts)
        g = W_ALL[np.flatnonzero(Y_dev[d])].max(0)

        def ev(idx):
            u = np.unique(np.concatenate([x[starts[j]: min(n, starts[j] + CHUNK_LEN)] for j in idx]))
            return float(g[u].sum())

        tot = ev(range(k))
        if tot <= 0:
            continue
        mid = [i for i in range(1, k - 1)]
        rnd = [0, k - 1] + [int(i) for i in rs.choice(mid, size=min(len(mid), max(SEL_K - 2, 0)), replace=False)]
        res["relevance (used)"].append(ev(SEL_REL["dev"][d]) / tot)
        res["evenly spaced"].append(ev(sorted(set(np.round(np.linspace(0, k - 1, SEL_K)).astype(int).tolist()))) / tot)
        res["random + first/last"].append(ev(sorted(set(rnd))) / tot)
        res["first + last only"].append(ev([0, k - 1]) / tot)
    out = {k_: float(np.mean(v)) for k_, v in res.items() if v}
    log(f"  gold-statute evidence kept by {SEL_K} windows (dev, {len(res['relevance (used)'])} long docs): " +
        " | ".join(f"{k_} {100 * v:.1f}%" for k_, v in out.items()))
    return out


log(f"SELECTIVE CHUNKING: mode={SEL_MODE}" + (f", K={SEL_K} (+first/last forced), relevance = information-gain evidence + citations" if SEL_MODE == "relevance" else ""))
SEL_REPORT = {"mode": SEL_MODE, "k": SEL_K, "summary": selection_summary(), "evidence_kept_dev": evidence_audit()}
atomic_json(SEL_REPORT, f"{WORK_DIR}/selection_report.json")

In [ ]:
# ============================================================
# CELL 7: BATCHING (token-budget, length-bucketed, SELECTED chunks) AND INFERENCE
# ============================================================
def doc_costs(split, sel):
    """real tokens a document costs in a batch = the tokens of the chunks that will be encoded (a document is never cut)"""
    st = stores[split]
    return np.array([sum(chunk_token_len(int(st.lens[d]), int(j)) for j in sel[d]) for d in range(len(st))], dtype=np.int64)


def pack(doc_order, costs, budget, max_docs):
    """greedy packing; a document is NEVER split or cut - one that exceeds the budget simply gets a batch of its own"""
    batches, cur, cost = [], [], 0
    for d in doc_order:
        c = int(costs[d])
        if cur and (cost + c > budget or len(cur) >= max_docs):
            batches.append(cur)
            cur, cost = [], 0
        cur.append(int(d))
        cost += c
    if cur:
        batches.append(cur)
    return batches


def make_train_batches(epoch):
    """deterministic given (SEED, epoch): shuffle -> sort by cost inside pools (less padding) -> pack -> shuffle batches.
    The documents are trained on their SELECTED chunks (SEL)."""
    costs = TRAIN_COSTS
    rng = np.random.RandomState(SEED + 1000 * epoch)
    order = rng.permutation(len(costs))
    POOL = 1536
    batches = []
    for p in range(0, len(order), POOL):
        pool = sorted(order[p:p + POOL], key=lambda d: costs[d])
        batches += pack(pool, costs, TRAIN_TOKEN_BUDGET, MAX_DOCS_PER_BATCH)
    perm = rng.permutation(len(batches))
    return [batches[i] for i in perm]


def make_eval_batches(split, sel):
    costs = doc_costs(split, sel)
    order = np.argsort(costs, kind="stable")
    return pack(order, costs, EVAL_TOKEN_BUDGET, 128)


def collate(split, doc_ids, sel):
    """the chunks sel[d] of every document in doc_ids -> padded ids + bookkeeping for the document head.
    owner/slot : which document / which slot (document order) the chunk fills ; opos : ORIGINAL window index ; ntot : windows in the document
    sf         : relevance score feature of the window"""
    store = stores[split]
    seqs, owner, slot, opos, ntot, sf = [], [], [], [], [], []
    for bi, d in enumerate(doc_ids):
        a = int(store.offsets[d])
        n = int(store.offsets[d + 1]) - a
        k_all = n_chunks_of(n)
        for si, j in enumerate(sel[d]):
            j = int(j)
            s = j * STRIDE
            seqs.append(store.tokens[a + s: a + min(n, s + CHUNK_LEN)])
            owner.append(bi)
            slot.append(si)
            opos.append(j)
            ntot.append(k_all)
            sf.append(SFEAT[split][d][j])
    lens = np.array([len(x) for x in seqs], dtype=np.int64)
    ids = np.full((len(seqs), int(lens.max())), PAD_ID, dtype=np.int64)
    for i, x in enumerate(seqs):
        ids[i, :len(x)] = x
    t = lambda a: torch.from_numpy(a).to(DEVICE, non_blocking=True)
    return {"ids": t(ids), "lens": t(lens), "owner": t(np.asarray(owner, dtype=np.int64)), "slot": t(np.asarray(slot, dtype=np.int64)),
            "opos": t(np.asarray(opos, dtype=np.int64)), "ntot": t(np.asarray(ntot, dtype=np.int64)),
            "sf": t(np.asarray(sf, dtype=np.float32)), "nd": len(doc_ids)}


@torch.no_grad()
def predict_probs(model, split, mode="selected", name="", paths=False):
    """sigmoid probabilities (n_docs, N_LABELS), rows aligned with the store.
    mode 'selected' = the relevance-selected chunks, 'all' = every chunk of every document.
    paths=True additionally returns {path name: probabilities} for the 3 aggregation paths (ablation)."""
    model.eval()
    store = stores[split]
    sel = sel_for(split, mode)
    P = np.zeros((len(store), N_LABELS), dtype=np.float32)
    PP = np.zeros((len(store), N_LABELS, 3), dtype=np.float32) if paths else None
    batches = make_eval_batches(split, sel)
    t0, done = time.time(), 0
    for i, docs in enumerate(batches):
        b = collate(split, docs, sel)
        with amp_ctx():
            logits, zs = model(b)
        P[docs] = torch.sigmoid(logits.float()).cpu().numpy()
        if paths:
            PP[docs] = torch.sigmoid(zs.float()).cpu().numpy()
        done += len(docs)
        if i % 50 == 0:
            log(f"  predict {name or split} ({mode}): {done:,}/{len(store):,} docs | {(time.time() - t0) / 60:.1f} min")
    if paths:
        return P, {PATH_NAMES[k]: PP[:, :, k] for k in range(3)}
    return P


TRAIN_COSTS = doc_costs("train", SEL["train"])
log(f"training cost per document: median {np.median(TRAIN_COSTS):.0f} tokens | p95 {np.percentile(TRAIN_COSTS, 95):.0f} | max {TRAIN_COSTS.max():,}")

In [ ]:
# ============================================================
# CELL 8: TRAINING  (LoRMA on the selected chunks; resumable, time-aware cosine LR, EMA, best epoch on dev macro-F1)
# Checkpoint files: lsi_latest.pt / lsi_latest_meta.json (resume) and lsi_best.pt / lsi_best_meta.json
# ============================================================
# pos_weight: mild up-weighting of positives for rare statutes (sqrt of the neg/pos ratio, capped)
_pos_cnt = Y_train.sum(0).astype(np.float64)
POS_W = torch.tensor(np.clip(np.sqrt((len(Y_train) - _pos_cnt) / np.maximum(_pos_cnt, 1)), 1.0, POS_WEIGHT_CAP),
                     dtype=torch.float32, device=DEVICE)
log(f"pos_weight range: {POS_W.min().item():.2f} .. {POS_W.max().item():.2f}")

Y_train_t = torch.from_numpy(Y_train.astype(np.float32))
TRAIN_EVAL_MODE = "selected" if SEL_MODE == "relevance" else "all"       # chunks used for the dev score during training

# everything that must be identical for a resumed run / a carried-over best checkpoint to be meaningful
CFG_SIG = hashlib.md5(json.dumps({
    "tok": tok.fingerprint, "labels": LABEL_IDS, "sel": [SEL_MODE, SEL_K, SEL_KEEP_FIRST, SEL_KEEP_LAST, SEL_FEAT, SEL_FOLDS, IG_SMOOTH, IG_MIN_DF, CITE_W],
    "chunk": [CHUNK_LEN, STRIDE], "ft": [FT_METHOD, LORA_R, LORA_ALPHA, LORMA_VARIANT if FT_METHOD == "lorma" else None, list(LORA_TARGETS)],
    "head": [HEAD_HIDDEN, DOC_LAYERS, DOC_HEADS, DOC_FF, MAX_POS_EMB, REL_BUCKETS], "smoke": SMOKE_TEST}, sort_keys=True).encode()).hexdigest()[:16]


def is_adapter(name):
    return name.endswith(".A") or name.endswith(".B")


def make_optimizer(model):
    groups = {"lora": [], "norm": [], "head_w": [], "head_nd": []}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("lm.") and is_adapter(n):
            groups["lora"].append(p)
        elif n.startswith("lm."):                       # enc_norm.weight
            groups["norm"].append(p)
        elif p.ndim >= 2:
            groups["head_w"].append(p)
        else:
            groups["head_nd"].append(p)
    wd_ad = 0.0 if (FT_METHOD == "lorma" and LORMA_VARIANT == "pi") else WEIGHT_DECAY      # LoRMA-pi: decay would pull the adapter away from the identity
    spec = [("lora", LORA_LR, wd_ad), ("norm", LORA_LR, 0.0), ("head_w", HEAD_LR, WEIGHT_DECAY), ("head_nd", HEAD_LR, 0.0)]
    pg = [{"params": groups[k], "lr": lr, "base_lr": lr, "weight_decay": wd, "adapter": (k == "lora")} for k, lr, wd in spec if groups[k]]
    return torch.optim.AdamW(pg, betas=(0.9, 0.99), eps=1e-8, fused=(DEVICE.type == "cuda"))


# ---------------------------- EMA over the FUNCTION the model computes ----------------------------
class ReLoRAEMA:
    """EMA of the effective weights (LoRA / rsLoRA: W + scale*B*A, LoRMA: M W0) of every adapted Linear and of the other trainable weights.
    Because it tracks the effective weight, it is unaffected by ReLoRA merges (a merge does not change the effective weight). `applied()`
    swaps it in (base weight := shadow, adapter switched off = identity) and restores everything afterwards."""

    def __init__(self, model, decay):
        self.decay = decay
        self.lora = lora_modules(model)
        self.others = [(n, p) for n, p in model.named_parameters() if p.requires_grad and not is_adapter(n)]
        self.sh_lora = [m.eff_weight().detach().float().clone() for _, m in self.lora]
        self.sh_other = [p.detach().float().clone() for _, p in self.others]

    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1.0 + step) / (10.0 + step))
        for s, (_, m) in zip(self.sh_lora, self.lora):
            s.mul_(d).add_(m.eff_weight().float(), alpha=1.0 - d)
        if self.sh_other:
            torch._foreach_mul_(self.sh_other, d)
            torch._foreach_add_(self.sh_other, [p.detach().float() for _, p in self.others], alpha=1.0 - d)

    @contextlib.contextmanager
    def applied(self):
        b_w = [m.base.weight.detach().clone() for _, m in self.lora]
        b_ad = [m.adapter_clone() for _, m in self.lora]
        b_o = [p.detach().clone() for _, p in self.others]
        with torch.no_grad():
            for s, (_, m) in zip(self.sh_lora, self.lora):
                m.base.weight.copy_(s)
                m.switch_off()
            for s, (_, p) in zip(self.sh_other, self.others):
                p.copy_(s)
        try:
            yield
        finally:
            with torch.no_grad():
                for w, ad, (_, m) in zip(b_w, b_ad, self.lora):
                    m.base.weight.copy_(w)
                    m.adapter_restore(ad)
                for o, (_, p) in zip(b_o, self.others):
                    p.copy_(o)

    def state_dict(self):
        return {"lora": {f"{n}.base.weight": s.detach().cpu().clone() for s, (n, _) in zip(self.sh_lora, self.lora)},
                "other": {n: s.detach().cpu().clone() for s, (n, _) in zip(self.sh_other, self.others)}}

    def load_state_dict(self, sd):
        for s, (n, _) in zip(self.sh_lora, self.lora):
            s.copy_(sd["lora"][f"{n}.base.weight"].to(s.device))
        for s, (n, _) in zip(self.sh_other, self.others):
            s.copy_(sd["other"][n].to(s.device))


# ---------------------------- ReLoRA restart ----------------------------
@torch.no_grad()
def reset_adapter_state(opt):
    """forget the Adam moments (and step count) of the adapters: the new adapters start from scratch"""
    n = 0
    for g in opt.param_groups:
        if not g.get("adapter"):
            continue
        for p in g["params"]:
            st = opt.state.get(p)
            if not st:
                continue
            for k, v in list(st.items()):
                if torch.is_tensor(v):
                    v.zero_()
                else:
                    st[k] = 0
            n += 1
    return n


@torch.no_grad()
def relora_restart(model, opt):
    """merge every adapter into its base weight (W += scale*B*A), re-initialise A, zero B, reset the adapters' optimizer state.
    The function computed by the model must not change - asserted."""
    mods = lora_modules(model)
    before = [m.eff_weight().clone() for _, m in mods]
    for _, m in mods:
        m.merge_and_reinit()
    err = max(float((m.eff_weight() - b).abs().max()) for (_, m), b in zip(mods, before))
    scale = max(float(b.abs().max()) for b in before)
    assert err <= 1e-5 * max(scale, 1.0), f"ReLoRA merge changed the model: max |dW| = {err:.3e}"
    n = reset_adapter_state(opt)
    del before
    return err, n


def lr_mults(step, prog, since_merge):
    """-> (multiplier for head/norm groups, multiplier for the adapter group).
    Both: linear warm-up (by steps) x cosine decay driven by `prog` = max(step progress, wall-clock progress).
    Adapters additionally re-warm from ~0 over RELORA_REWARM steps after every merge (jagged cosine)."""
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    base = warm * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))
    rew = 1.0 if since_merge is None else min(1.0, (since_merge + 1) / max(RELORA_REWARM, 1))
    return base, base * rew


def find_previous_latest():
    best = None
    for d in walk_find("lsi_latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_latest_meta.json"))
        if not meta or meta.get("tag") != RUN_TAG or meta.get("cfg_sig") != CFG_SIG:
            continue
        if not os.path.exists(os.path.join(d, "lsi_latest.pt")):
            continue
        if best is None or meta["global_step"] > best[1]["global_step"]:
            best = (d, meta)
    return best


def carry_over_best():
    """copy the best checkpoint of an earlier session (attached as input) into this session's checkpoint dir"""
    cur = read_json(f"{CKPT_DIR}/lsi_best_meta.json")
    for d in walk_find("lsi_best_meta.json", [INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_best_meta.json"))
        if not meta or meta.get("tag") != RUN_TAG or meta.get("cfg_sig") != CFG_SIG or not os.path.exists(os.path.join(d, "lsi_best.pt")):
            continue
        if cur is None or meta["dev_macro_f1"] > cur["dev_macro_f1"]:
            shutil.copy(os.path.join(d, "lsi_best.pt"), f"{CKPT_DIR}/lsi_best.pt.tmp")
            os.replace(f"{CKPT_DIR}/lsi_best.pt.tmp", f"{CKPT_DIR}/lsi_best.pt")
            shutil.copy(os.path.join(d, "lsi_best_meta.json"), f"{CKPT_DIR}/lsi_best_meta.json")
            cur = meta
    return cur


def dev_eval(model):
    """dev macro-F1 with the best single global threshold (cheap, comparable across epochs)"""
    P = predict_probs(model, "dev", TRAIN_EVAL_MODE, "dev")
    thr = fit_thresholds(P, Y_dev, "global")
    m = all_metrics(Y_dev, apply_thresholds(P, thr))
    log(f"  DEV  macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | subset-acc {m['subset_acc']:.4f} "
        f"| global thr {thr[0]:.2f} | avg labels/doc {m['avg_pred_labels']:.2f}  ({TRAIN_EVAL_MODE} chunks)")
    return m


def train(model):
    best_meta = carry_over_best()
    best_f1 = best_meta["dev_macro_f1"] if best_meta else -1.0
    opt = make_optimizer(model)
    scaler = make_grad_scaler(AMP["enabled"] and AMP["dtype"] == torch.float16)
    state = {"epoch": 0, "bi": 0, "global_step": 0, "train_sec": 0.0, "last_merge": 0, "n_merges": 0}
    ck_ema = None

    prev = find_previous_latest()
    if prev is not None:
        d, meta = prev
        ck = torch.load(os.path.join(d, "lsi_latest.pt"), map_location="cpu", weights_only=False)
        load_changing(model, ck["changing"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        state.update(epoch=ck["epoch"], bi=ck["bi"], global_step=ck["global_step"], train_sec=ck.get("train_sec", 0.0),
                     last_merge=ck.get("last_merge", 0), n_merges=ck.get("n_merges", 0))
        ck_ema = ck.get("ema")
        log(f"RESUMED from {d}: epoch {state['epoch'] + 1}, batch {state['bi']}, step {state['global_step']}, "
            f"{state['n_merges']} adapter merges so far, train time so far {fmt_time(state['train_sec'])}")
        del ck
    ema = ReLoRAEMA(model, EMA_DECAY) if USE_EMA else None
    if ema is not None and ck_ema is not None:
        ema.load_state_dict(ck_ema)
    train_params = [p for g in opt.param_groups for p in g["params"]]

    n_batches0 = len(make_train_batches(0))
    total_steps = EPOCHS * n_batches0
    n_chunks_train = int(sum(len(x) for x in SEL["train"]))
    log(f"{len(stores['train']):,} train docs | {n_chunks_train:,} chunks per epoch ({SEL_MODE}) | ~{n_batches0:,} steps/epoch "
        f"| {EPOCHS} epochs | ~{total_steps:,} steps | {ft_label()} scale {lora_scale_of(model):.3f}"
        + (f" | ReLoRA: merge every {RELORA_EVERY} steps (~{total_steps // RELORA_EVERY} merges)" if RELORA else ""))

    # time-aware schedule: the LR reaches its minimum when EPOCHS finish OR the training-time budget is used up
    sess_avail = (TIME_BUDGET_H - FINAL_RESERVE_H) * 3600 - (time.time() - T_START)
    train_budget = min(TRAIN_HOURS * 3600, state["train_sec"] + max(sess_avail, 0.0))
    log(f"training-time budget: {fmt_time(train_budget)} in total ({fmt_time(state['train_sec'])} already used) | "
        f"session wall-clock limit {TIME_BUDGET_H}h, {FINAL_RESERVE_H}h reserved for final prediction")

    last = [time.time()]

    def tick():
        now = time.time()
        state["train_sec"] += now - last[0]
        last[0] = now

    def progress():
        return min(1.0, max(state["global_step"] / total_steps, state["train_sec"] / max(train_budget, 1.0)))

    def save_latest():
        atomic_torch_save({"changing": changing_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                           "ema": ema.state_dict() if ema is not None else None,
                           "epoch": state["epoch"], "bi": state["bi"], "global_step": state["global_step"],
                           "train_sec": state["train_sec"], "last_merge": state["last_merge"], "n_merges": state["n_merges"],
                           "tag": RUN_TAG}, f"{CKPT_DIR}/lsi_latest.pt")
        atomic_json({"tag": RUN_TAG, "cfg_sig": CFG_SIG, "n_labels": N_LABELS, "epoch": state["epoch"], "bi": state["bi"],
                     "global_step": state["global_step"], "train_sec": state["train_sec"], "n_merges": state["n_merges"],
                     "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f"{CKPT_DIR}/lsi_latest_meta.json")

    def eval_and_maybe_save_best(tag):
        nonlocal best_f1
        with (ema.applied() if ema is not None else contextlib.nullcontext()):      # dev score of the EMA weights
            m = dev_eval(model)
            if m["macro_f1"] > best_f1:
                best_f1 = m["macro_f1"]
                atomic_torch_save({"merged": merged_weights(model), "dev_macro_f1": best_f1, "tag": RUN_TAG}, f"{CKPT_DIR}/lsi_best.pt")
                atomic_json({"tag": RUN_TAG, "cfg_sig": CFG_SIG, "dev_macro_f1": best_f1, "where": tag, "global_step": state["global_step"],
                             "n_merges": state["n_merges"], "label_ids": LABEL_IDS, "n_labels": N_LABELS, "tok_fp": tok.fingerprint},
                            f"{CKPT_DIR}/lsi_best_meta.json")
                log(f"  >> NEW BEST dev macro-F1 {best_f1:.4f} ({tag}) - lsi_best.pt saved")
        return m

    timed_out = False
    t_loop = time.time()
    steps_this_run, chunks_seen, tokens_seen, run_loss, run_n, n_skipped = 0, 0, 0, 0.0, 0, 0

    for epoch in range(state["epoch"], EPOCHS):
        batches = make_train_batches(epoch)
        model.train()
        for bi in range(state["bi"], len(batches)):
            if state["train_sec"] >= train_budget:
                timed_out = True
                break
            docs = batches[bi]
            b = collate("train", docs, SEL["train"])                         # the SELECTED chunks of every doc in the batch
            y = Y_train_t[docs].to(DEVICE)

            since = (state["global_step"] - state["last_merge"]) if state["n_merges"] > 0 else None
            m_head, m_lora = lr_mults(state["global_step"], progress(), since)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * (m_lora if g.get("adapter") else m_head)

            with amp_ctx():
                logits, zs = model(b)

            def bce(z):
                return F.binary_cross_entropy_with_logits(z.float(), y, pos_weight=POS_W, reduction="none").mean(1)

            loss_main = bce(logits)
            loss_aux = sum(bce(zs[:, :, k]) for k in range(3)) / 3.0         # every path must be useful on its own
            loss = (loss_main + AUX_W * loss_aux).sum() / DOC_NORM
            opt.zero_grad(set_to_none=True)
            if torch.isfinite(loss):
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(train_params, GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
                if ema is not None:
                    ema.update(state["global_step"])
                run_loss += loss_main.mean().item()
                run_n += 1
            else:
                n_skipped += 1
                log(f"  WARNING: non-finite loss at step {state['global_step']} (batch of {len(docs)} docs) - step skipped")

            state["global_step"] += 1
            state["epoch"], state["bi"] = epoch, bi + 1
            steps_this_run += 1
            chunks_seen += int(b["lens"].numel())
            tokens_seen += int(b["lens"].sum().item())
            tick()

            # ---- ReLoRA: fold the adapters into the weights and start new ones (not in the last part of the schedule) ----
            if RELORA and state["global_step"] - state["last_merge"] >= RELORA_EVERY and progress() < RELORA_STOP_PROG:
                err, n_opt = relora_restart(model, opt)
                state["last_merge"] = state["global_step"]
                state["n_merges"] += 1
                log(f"  ReLoRA merge #{state['n_merges']} at step {state['global_step']} | effective weights unchanged (max |dW| {err:.1e}) "
                    f"| adapter optimizer state reset ({n_opt} tensors) | cumulative adapter rank <= {LORA_R * (state['n_merges'] + 1)}")

            if steps_this_run % LOG_EVERY == 0:
                el = time.time() - t_loop
                eta_epoch = (len(batches) - bi - 1) * el / steps_this_run
                log(f"ep {epoch + 1}/{EPOCHS} step {bi + 1}/{len(batches)} (global {state['global_step']}) | "
                    f"loss {run_loss / max(run_n, 1):.4f} | lr x{m_head:.3f} (adapters x{m_lora:.3f}) | {chunks_seen / el:.1f} chunks/s "
                    f"{tokens_seen / el / 1000:.1f}k tok/s | epoch ETA {fmt_time(eta_epoch)} | "
                    f"schedule {100 * progress():.0f}% | train time {fmt_time(state['train_sec'])}/{fmt_time(train_budget)}")
                run_loss, run_n = 0.0, 0
            if state["global_step"] % SAVE_EVERY == 0:
                save_latest()
                tick()

        if timed_out:
            log(f"training-time budget reached inside epoch {epoch + 1} - saving, then dev tuning / prediction")
            save_latest()
            eval_and_maybe_save_best(f"timeout@epoch{epoch + 1}")
            tick()
            break
        state["epoch"], state["bi"] = epoch + 1, 0
        save_latest()
        eval_and_maybe_save_best(f"epoch{epoch + 1}")
        tick()
        model.train()

    if n_skipped:
        log(f"note: {n_skipped} step(s) skipped because of a non-finite loss")
    log(f"training finished | best dev macro-F1 (global thr) {best_f1:.4f} | {state['n_merges']} adapter merges | train time {fmt_time(state['train_sec'])} "
        f"| session elapsed {fmt_time(time.time() - T_START)}")
    return best_f1


model = build_model()
BEST_DEV_F1 = train(model)

In [ ]:
# ============================================================
# CELL 9: BEST CHECKPOINT -> PATH ABLATION -> SELECTED vs ALL CHUNKS -> DEV THRESHOLD TUNING -> TEST PREDICTION -> submission.zip
# ============================================================
bm = read_json(f"{CKPT_DIR}/lsi_best_meta.json")
assert bm is not None and os.path.exists(f"{CKPT_DIR}/lsi_best.pt"), "no best checkpoint found - did training run?"
assert bm["label_ids"] == LABEL_IDS, "label space differs from the one the checkpoint was trained with"
assert bm.get("tok_fp") in (None, tok.fingerprint), "the best checkpoint was trained with another tokenizer"
ck = torch.load(f"{CKPT_DIR}/lsi_best.pt", map_location="cpu", weights_only=False)
load_merged(model, ck["merged"])                    # merged effective weights (EMA), adapters switched off
log(f"loaded best checkpoint (dev macro-F1 {ck['dev_macro_f1']:.4f}, {bm['where']}, after {bm.get('n_merges', 0)} adapter merges)")
del ck
gc.collect()

# ---- which chunks are scored at prediction time ----
if SEL_MODE == "all":
    CAND_MODES = ["all"]                                          # trained on all chunks -> predict on all chunks
elif FINAL_CHUNKS == "auto":
    CAND_MODES = ["selected", "all", "both"]
else:
    CAND_MODES = [FINAL_CHUNKS]
need = ["selected"] if "selected" in CAND_MODES or "both" in CAND_MODES else []
need += ["all"] if "all" in CAND_MODES or "both" in CAND_MODES else []

# ---- 1) dev probabilities for every needed chunk view (the first one also gives the 3 aggregation paths) ----
PD, PD_paths = {}, None
for mode in need:
    if PD_paths is None:
        PD[mode], PD_paths = predict_probs(model, "dev", mode, "dev", paths=True)
        PD_paths_mode = mode
    else:
        PD[mode] = predict_probs(model, "dev", mode, "dev")
if "both" in CAND_MODES:
    PD["both"] = 0.5 * (PD["selected"] + PD["all"])
for k_, v_ in PD.items():
    assert np.isfinite(v_).all(), f"non-finite dev probabilities ({k_})"

log(f"\nAggregation-path ablation on dev ({PD_paths_mode} chunks; global threshold, >=1 label/doc; paths were trained jointly):")
for name, Pp in list(PD_paths.items()) + [("FUSED (used)", PD[PD_paths_mode])]:
    thr_g = fit_thresholds(Pp, Y_dev, "global")
    m = all_metrics(Y_dev, apply_thresholds(Pp, thr_g))
    log(f"  {name:13s} macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | subset-acc {m['subset_acc']:.4f}")

# ---- 2) honest cross-fitted dev estimate per chunk view; the best view is used for the test set ----
CF = {mode: crossfit_report(PD[mode], Y_dev) for mode in CAND_MODES}
SCORE_OF = {mode: max(v["macro_f1"] for v in CF[mode].values()) for mode in CAND_MODES}
log("\nDEV, cross-fitted estimate (thresholds fitted on one half of dev, scored on the other; force >=1 label per document):")
for mode in CAND_MODES:
    desc = {"selected": f"selected chunks (<= {SEL_K}/doc)", "all": "ALL chunks", "both": "average of selected + all"}[mode]
    for tmode, m in CF[mode].items():
        log(f"  {desc:28s} {tmode:9s} macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | "
            f"subset-acc {m['subset_acc']:.4f} | avg labels/doc {m['avg_pred_labels']:.2f}")
FINAL_VIEW = max(CAND_MODES, key=lambda k: (SCORE_OF[k], k == "selected"))         # ties -> the cheaper selected view
if len(CAND_MODES) > 1:
    log("chunk view used for the submission: " + FINAL_VIEW + "  (" + " | ".join(f"{k_} {SCORE_OF[k_]:.4f}" for k_ in CAND_MODES) + ")")

P_dev = PD[FINAL_VIEW]
BEST_MODE = max(CF[FINAL_VIEW], key=lambda k: CF[FINAL_VIEW][k]["macro_f1"])
THR = fit_thresholds(P_dev, Y_dev, BEST_MODE)          # final thresholds use all of dev
pred_dev = apply_thresholds(P_dev, THR)
m_dev = all_metrics(Y_dev, pred_dev)
log(f"chosen threshold strategy: {BEST_MODE} | in-sample dev macro-F1 {m_dev['macro_f1']:.4f} (optimistic, as thresholds were fitted on it)")
log(f"thresholds: min {THR.min():.2f} | median {np.median(THR):.2f} | max {THR.max():.2f}")

f1_l = f1_per_label(Y_dev, pred_dev)
worst = np.argsort(f1_l)[:8]
log("weakest statutes on dev (id: F1, train positives): " +
    ", ".join(f"{LABEL_IDS[i]}: {f1_l[i]:.2f} ({int(Y_train[:, i].sum())})" for i in worst))

# ---- 3) test prediction ----
PT = {}
for mode in need:
    if mode == FINAL_VIEW or FINAL_VIEW == "both":
        PT[mode] = predict_probs(model, "test", mode, "test")
P_test = 0.5 * (PT["selected"] + PT["all"]) if FINAL_VIEW == "both" else PT[FINAL_VIEW]
assert np.isfinite(P_test).all(), "non-finite test probabilities"
pred_test = apply_thresholds(P_test, THR)

preds = []
valid_ids = set(LABEL_IDS)
for i, doc_id in enumerate(TEST_IDS):
    ids = sorted({int(LABEL_IDS[j]) for j in np.where(pred_test[i])[0]})       # unique ints, valid statute ids only
    assert ids and all(x in valid_ids for x in ids)
    preds.append({"id": doc_id, "pred": ids})

assert len(preds) == len(TEST_IDS) == len(stores["test"])
assert len({str(p["id"]) for p in preds}) == len(preds), "duplicate test ids"
log(f"test: {len(preds):,} predictions | avg statutes/doc {np.mean([len(p['pred']) for p in preds]):.2f} "
    f"(dev truth: {Y_dev.sum(1).mean():.2f})")

# ---- 4) save everything ----
pred_path = f"{OUT_DIR}/prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)
zip_path = f"{OUT_DIR}/submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")            # prediction.json at the zip root

np.save(f"{OUT_DIR}/dev_probs.npy", P_dev)                   # keep for ensembling with the other Phase-1 models
np.save(f"{OUT_DIR}/test_probs.npy", P_test)
atomic_json({"label_ids": LABEL_IDS, "thresholds": [float(x) for x in THR], "strategy": BEST_MODE, "chunk_view": FINAL_VIEW,
             "dev_crossfit_by_view": CF, "dev_in_sample": m_dev, "test_ids": [str(x) for x in TEST_IDS],
             "tokenizer": {"kind": tok.kind, "fingerprint": tok.fingerprint, "where": tok.where}, "run_tag": RUN_TAG,
             "selection": SEL_REPORT,
             "lora": {"method": ft_label(), "ft_method": FT_METHOD, "variant": LORMA_VARIANT if FT_METHOD == "lorma" else None, "rank": LORA_R,
                      "alpha": None if (FT_METHOD == "lorma" and LORMA_VARIANT == "pi") else LORA_ALPHA, "scale": lora_scale_of(model),
                      "targets": list(LORA_TARGETS)},
             "relora": {"enabled": RELORA, "every": RELORA_EVERY, "rank": LORA_R, "merges_at_best": bm.get("n_merges", 0)},
             "best_checkpoint": {"dev_macro_f1_global_thr": bm["dev_macro_f1"], "where": bm["where"], "global_step": bm["global_step"]}},
            f"{OUT_DIR}/thresholds_and_report.json")
log("saved:", pred_path, "|", zip_path, "| dev_probs.npy | test_probs.npy | thresholds_and_report.json")
log("done.")